<a href="https://colab.research.google.com/github/ClaraMourgueDAlgue/QM2/blob/main/Copy_of_qf209_a4_person2_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QF209 A4 Smart Predict-then-Optimize | Sprint 2 — Person 2 baseline
**Owner: Person 2 · OLS/ridge forecasts + mean–variance optimisation + baseline diagnosis**

Inputs: `train_panel.csv`, `validation_panel.csv`, `validation_benchmark_metrics.csv`, `iteration_log.csv` from Person 1's package (put them next to this notebook, or upload them in Colab). **No test labels are used. The test set stays locked.**

Design choices (all fixed before looking at validation):
- **Forecast:** one pooled linear model for all 20 ETFs, using the 9 features z-scored on the fitting window. α = 0 is OLS; α > 0 is ridge (intercept not penalised). Fitted once on all 2010–2018 training weeks, then frozen for validation (no rolling retrain in the baseline).
- **Risk:** sample covariance of the last 156 weekly returns (`ret_1w` up to and including the decision Friday, so only information known at t).
- **Optimiser:** max μᵀw − ½γ wᵀΣw subject to Σw = 1, Σ|w| ≤ 1.5, |wᵢ| ≤ 1. Solved with scipy SLSQP using the split w = u − v; no cvxpy needed.
- **Tuning on the 4 internal folds only:** α by lowest fold MSE (the "accuracy" step), then γ by highest mean fold net Sharpe given that α.
- **Scoring:** Person 1's `evaluate_portfolio`, copied verbatim (5 bp, rf = 0).

In [6]:
# Colab: upload the CSVs first, e.g.
# from google.colab import files; files.upload()
import os; os.makedirs("fig", exist_ok=True)


## 1. Core functions (data loading, shared backtester, forecaster, covariance, optimiser)

In [7]:
"""QF209 A4 - Person 2: OLS/ridge predict-then-optimise baseline.
Uses only train_panel.csv and validation_panel.csv (no test labels).
"""
import numpy as np, pandas as pd
from scipy.optimize import minimize

TICKERS = ['SPY','XLK','XLF','XLE','XLV','XLI','XLY','XLP','XLU','XLB',
           'QQQ','IWM','EFA','EEM','TLT','IEF','LQD','HYG','GLD','VNQ']
FEATURES = ['ret_1w','mom_4w','mom_12w','mom_26w','vol_12w','vol_26w',
            'spy_mom_4w','spy_mom_12w','spy_vol_12w']
MAX_GROSS, MAX_W, COST_BPS = 1.5, 1.0, 5.0
COV_WINDOW = 156  # weeks of past realised returns for the covariance estimate


def load_panels(train_path='train_panel.csv', val_path='validation_panel.csv'):
    out = {}
    for name, p in [('train', train_path), ('validation', val_path)]:
        df = pd.read_csv(p, parse_dates=['decision_date', 'target_end_date'])
        out[name] = df.set_index(['decision_date', 'ticker']).sort_index()
    panel = pd.concat([out['train'], out['validation']]).sort_index()
    split_dates = {k: v.index.get_level_values(0).unique().sort_values() for k, v in out.items()}
    targets = panel['target_return'].unstack('ticker')[TICKERS]
    # ret_1w at decision date t is the return of the week ENDING at t -> known at t
    past_returns = panel['ret_1w'].unstack('ticker')[TICKERS]
    return panel, split_dates, targets, past_returns


# ---------------- shared backtester (verbatim from Person 1's notebook) ----------------
def evaluate_portfolio(weights, forward_returns, predictions=None,
                       cost_bps=5.0, rf_annual=0.0,
                       allow_short=True, max_gross=1.5, max_weight_abs=1.0):
    if not isinstance(weights, pd.DataFrame) or not isinstance(forward_returns, pd.DataFrame):
        raise TypeError('weights and forward_returns must be pandas DataFrames')
    if weights.empty: raise ValueError('Weights are empty.')
    tickers = list(forward_returns.columns)
    if set(weights.columns) != set(tickers):
        raise ValueError('Weight columns must exactly equal forward return tickers.')
    if not weights.index.is_unique or not weights.index.is_monotonic_increasing:
        raise ValueError('Weight dates must be unique and increasing.')
    if any(date not in forward_returns.index for date in weights.index):
        raise ValueError('A weight date is missing from forward_returns.')
    if not (weights.index.to_series().diff().dropna() == pd.Timedelta(days=7)).all():
        raise ValueError('Weight dates must be consecutive Fridays; otherwise missed weekly P&L is hidden.')
    wdf = weights.reindex(columns=tickers).astype(float)
    rdf = forward_returns.reindex(index=wdf.index, columns=tickers).astype(float)
    if not (np.isfinite(wdf.to_numpy()).all() and np.isfinite(rdf.to_numpy()).all()):
        raise ValueError('Nonfinite weights or actual returns.')
    if not np.allclose(wdf.sum(axis=1).values, 1.0, atol=1e-6):
        raise ValueError('Every portfolio row must sum to 1.0 (fully invested).')
    if (wdf.abs().sum(axis=1) > max_gross + 1e-6).any():
        raise ValueError('Weights breach the gross leverage limit.')
    if (wdf.abs() > max_weight_abs + 1e-6).any().any():
        raise ValueError('Weights breach per-asset absolute-weight limit.')
    if not allow_short and (wdf < -1e-8).any().any():
        raise ValueError('Negative weights not allowed in long-only experiment.')
    if not 0 <= cost_bps <= 10000: raise ValueError('Invalid cost in basis points.')
    pre_trade = np.zeros(len(tickers), dtype=float)
    records = []
    fee_rate = cost_bps / 10000.0
    for date in wdf.index:
        target = wdf.loc[date].to_numpy(dtype=float)
        r = rdf.loc[date].to_numpy(dtype=float)
        traded_notional = float(np.abs(target - pre_trade).sum())
        cost_fraction = fee_rate * traded_notional
        gross_return = float(np.dot(target, r))
        if 1 + gross_return <= 0 or cost_fraction >= 1:
            raise ValueError(f'Invalid portfolio wealth factor at {date}.')
        net_return = (1 - cost_fraction) * (1 + gross_return) - 1
        records.append({'decision_date': date, 'gross_return': gross_return,
                        'net_return': net_return, 'turnover': traded_notional,
                        'cost_fraction': cost_fraction,
                        'gross_leverage': float(np.abs(target).sum())})
        pre_trade = target * (1 + r) / (1 + gross_return)
    series = pd.DataFrame.from_records(records).set_index('decision_date')
    if predictions is not None:
        if set(predictions.columns) != set(tickers):
            raise ValueError('Prediction columns must match asset universe.')
        if not wdf.index.isin(predictions.index).all():
            raise ValueError('Missing prediction dates.')
        pred = predictions.reindex(index=wdf.index, columns=tickers).astype(float)
        if not np.isfinite(pred.to_numpy()).all():
            raise ValueError('Predictions contain missing/nonfinite values.')
        mse = float(np.mean((pred.to_numpy() - rdf.to_numpy()) ** 2))
    else:
        mse = float('nan')
    rf_weekly = (1 + rf_annual) ** (1 / 52) - 1
    excess = series['net_return'] - rf_weekly
    sd = float(excess.std(ddof=1))
    sharpe = float(np.sqrt(52) * excess.mean() / sd) if sd > 0 else float('nan')
    wealth = pd.concat([pd.Series([1.0], index=[wdf.index[0] - pd.Timedelta(days=7)]),
                        (1 + series['net_return']).cumprod()]).sort_index()
    max_dd = float((wealth / wealth.cummax() - 1).min())
    n = len(series)
    summary = {
        'n_weeks': n,
        'ann_return': float((1 + series['net_return']).prod() ** (52 / n) - 1),
        'ann_volatility': float(series['net_return'].std(ddof=1) * np.sqrt(52)),
        'sharpe_net': sharpe,
        'max_drawdown': max_dd,
        'mean_weekly_turnover': float(series['turnover'].mean()),
        'total_traded_notional': float(series['turnover'].sum()),
        'total_cost_fraction_sum': float(series['cost_fraction'].sum()),
        'prediction_mse': mse,
        'cost_bps': cost_bps,
    }
    return summary, series, wealth


# ---------------- 1. forecasting model ----------------
class PooledLinearForecaster:
    """One pooled linear model across all 20 ETFs. alpha=0 -> OLS, alpha>0 -> ridge.
    Features are z-scored with mean/std from the fitting window only; intercept not penalised."""
    def __init__(self, alpha=0.0):
        self.alpha = float(alpha)

    def fit(self, panel, dates):
        d = panel.loc[panel.index.get_level_values(0).isin(dates)]
        X = d[FEATURES].to_numpy(float); y = d['target_return'].to_numpy(float)
        self.mu_, self.sd_ = X.mean(0), X.std(0, ddof=0)
        Z = (X - self.mu_) / self.sd_
        self.b0_ = y.mean()
        yc = y - self.b0_
        self.coef_ = np.linalg.solve(Z.T @ Z + self.alpha * np.eye(Z.shape[1]), Z.T @ yc)
        return self

    def predict(self, panel, dates):
        d = panel.loc[panel.index.get_level_values(0).isin(dates)]
        Z = (d[FEATURES].to_numpy(float) - self.mu_) / self.sd_
        pred = pd.Series(self.b0_ + Z @ self.coef_, index=d.index)
        return pred.unstack('ticker')[TICKERS]


# ---------------- 2. covariance estimate (past information only) ----------------
def rolling_cov(past_returns, date, window=COV_WINDOW):
    hist = past_returns.loc[:date].tail(window)  # includes week ending at t, never t+1
    if len(hist) < 52:
        raise ValueError(f'Not enough history for covariance at {date}')
    return hist.cov().to_numpy()


# ---------------- 3. constrained mean-variance optimiser ----------------
def mv_weights(mu, Sigma, gamma, w0=None, max_gross=MAX_GROSS, max_w=MAX_W, allow_short=True):
    """max mu'w - gamma/2 w'Sigma w  s.t. sum w = 1, sum|w| <= max_gross, |w_i| <= max_w.
    Split w = u - v with u, v >= 0 so the L1 constraint is linear (exact at optimum)."""
    n = len(mu); S = 1e3  # scale objective for solver tolerance
    if not allow_short:
        bounds = [(0, max_w)] * n + [(0, 0)] * n
    else:
        bounds = [(0, max_w)] * (2 * n)

    def f(x):
        w = x[:n] - x[n:]
        return -S * (mu @ w - 0.5 * gamma * w @ Sigma @ w)

    def g(x):
        w = x[:n] - x[n:]
        gw = -S * (mu - gamma * Sigma @ w)
        return np.concatenate([gw, -gw])

    cons = [{'type': 'eq', 'fun': lambda x: x[:n].sum() - x[n:].sum() - 1.0,
             'jac': lambda x: np.concatenate([np.ones(n), -np.ones(n)])},
            {'type': 'ineq', 'fun': lambda x: max_gross - x.sum(),
             'jac': lambda x: -np.ones(2 * n)}]
    if w0 is None:
        w0 = np.full(n, 1.0 / n)
    x0 = np.concatenate([np.clip(w0, 0, None), np.clip(-w0, 0, None)])
    res = minimize(f, x0, jac=g, bounds=bounds, constraints=cons, method='SLSQP',
                   options={'maxiter': 500, 'ftol': 1e-12})
    w = res.x[:n] - res.x[n:]
    # tidy numerical noise so the strict backtester checks pass
    w[np.abs(w) < 1e-9] = 0.0
    w = np.clip(w, -max_w, max_w)
    w = w + (1.0 - w.sum()) / n
    if np.abs(w).sum() > max_gross:  # rare: rescale shorts slightly
        neg = w < 0; excess = np.abs(w).sum() - max_gross
        w[neg] *= 1 - excess / (2 * np.abs(w[neg]).sum())
        w[~neg] *= (1 - w[neg].sum()) / w[~neg].sum()
    return w, res.success


def build_weights(pred, past_returns, dates, gamma, **kw):
    rows, w_prev, fails = [], None, 0
    for t in dates:
        Sigma = rolling_cov(past_returns, t)
        w, ok = mv_weights(pred.loc[t, TICKERS].to_numpy(), Sigma, gamma, w0=w_prev, **kw)
        fails += (not ok); rows.append(w); w_prev = w
    return pd.DataFrame(rows, index=dates, columns=TICKERS), fails


def run_strategy(pred, targets, past_returns, dates, gamma, allow_short=True):
    W, fails = build_weights(pred, past_returns, dates, gamma, allow_short=allow_short)
    s, series, wealth = evaluate_portfolio(W, targets.loc[dates, TICKERS],
                                           predictions=pred.loc[dates, TICKERS],
                                           cost_bps=COST_BPS, allow_short=allow_short,
                                           max_gross=MAX_GROSS, max_weight_abs=MAX_W)
    s['solver_failures'] = fails
    return s, series, wealth, W


def r2_oos(pred, actual, benchmark=0.0):
    p = pred.to_numpy().ravel(); a = actual.to_numpy().ravel()
    return 1 - np.sum((a - p) ** 2) / np.sum((a - benchmark) ** 2)


In [8]:
panel, SPLIT, TARGETS, PAST = load_panels('train_panel.csv', 'validation_panel.csv')
print({k: len(v) for k, v in SPLIT.items()}, 'weeks')  # expect train 470, validation 157


{'train': 470, 'validation': 157} weeks


## 2. Tuning on the four internal folds (2016–2018 only)

In [9]:
import time, numpy as np, pandas as pd, itertools
tr = SPLIT['train']
# fold schedule: replicate Person 1 (156w train, 26w val, step 26, last 4) on decision dates
n=len(tr); folds=[]
for e in range(156, n-26+1, 26): folds.append((tr[e-156:e], tr[e:e+26]))
folds=folds[-4:]
for i,(a,b) in enumerate(folds,1):
    print(i, (a[0]+pd.Timedelta(days=7)).date(), (a[-1]+pd.Timedelta(days=7)).date(), (b[0]+pd.Timedelta(days=7)).date(), (b[-1]+pd.Timedelta(days=7)).date())
alphas=[0,10,100,1000,3000,10000,30000,100000]
gammas=[1,2.5,5,10,25,50,100]
rows=[]; t0=time.time()
for a in alphas:
    for k,(ftr,fva) in enumerate(folds,1):
        m=PooledLinearForecaster(a).fit(panel, ftr)
        pred=m.predict(panel, fva)
        act=TARGETS.loc[fva]
        mse=float(((pred-act)**2).to_numpy().mean())
        for g in gammas:
            s,_,_,W=run_strategy(pred, TARGETS, PAST, fva, g)
            rows.append(dict(alpha=a,gamma=g,fold=k,mse=mse,r2_zero=r2_oos(pred,act),sharpe=s['sharpe_net'],
                             turnover=s['mean_weekly_turnover'],fails=s['solver_failures']))
    print(a, round(time.time()-t0,1))
R=pd.DataFrame(rows); R.to_csv('tuning_folds.csv',index=False)
print(R.groupby('alpha')[['mse','r2_zero']].mean())
print(R.groupby(['alpha','gamma'])['sharpe'].mean().unstack().round(2))
print('fails', R.fails.sum())


1 2013-12-27 2016-12-16 2016-12-23 2017-06-16
2 2014-06-27 2017-06-16 2017-06-23 2017-12-15
3 2014-12-26 2017-12-15 2017-12-22 2018-06-15
4 2015-06-26 2018-06-15 2018-06-22 2018-12-14
0 16.7
10 33.6
100 50.3
1000 70.5
3000 88.7
10000 107.0
30000 129.7
100000 149.8
             mse   r2_zero
alpha                     
0       0.000329 -0.014956
10      0.000329 -0.014771
100     0.000328 -0.013205
1000    0.000327 -0.003413
3000    0.000326  0.004866
10000   0.000325  0.011927
30000   0.000325  0.014771
100000  0.000326  0.015550
gamma   1.0    2.5    5.0    10.0   25.0   50.0   100.0
alpha                                                  
0        0.26   0.31   0.30   0.49   0.55   0.67   0.85
10       0.27   0.31   0.30   0.49   0.56   0.67   0.85
100      0.29   0.31   0.29   0.51   0.58   0.69   0.87
1000     0.46   0.50   0.71   0.72   0.80   0.88   0.98
3000     0.83   0.88   1.03   1.01   0.96   1.05   1.04
10000    0.67   1.17   1.20   1.06   1.07   1.13   1.04
30000    1.00   1

## 3. Final fit on 2010–2018, evaluated on 2019–2021 validation, plus references and diagnostics
The α × γ grid on validation is a **diagnostic** of how far MSE and Sharpe diverge. It is not used to pick the model.

In [10]:
import numpy as np, pandas as pd, json
from scipy.stats import spearmanr
tr, va = SPLIT['train'], SPLIT['validation']
act = TARGETS.loc[va, TICKERS]
ALPHA, GAMMA = 10000, 5.0          # chosen on internal folds (MSE for alpha, Sharpe for gamma)
OLS_GAMMA = 100.0                  # best fold Sharpe for OLS (grid edge)

res, store = {}, {}
def record(name, pred, gamma, allow_short=True):
    s, series, wealth, W = run_strategy(pred, TARGETS, PAST, va, gamma, allow_short)
    s['r2_vs_zero'] = r2_oos(pred, act); s['gamma']=gamma
    ic = [spearmanr(pred.loc[t], act.loc[t]).statistic for t in va]
    s['mean_rank_ic'] = float(np.nanmean(ic))
    s['gross_sharpe'] = float(np.sqrt(52)*series.gross_return.mean()/series.gross_return.std())
    s['mean_gross_leverage'] = float(series.gross_leverage.mean())
    s['share_weeks_at_lev_cap'] = float((series.gross_leverage>1.5-1e-3).mean())
    s['mean_short_exposure'] = float(W.clip(upper=0).sum(1).abs().mean())
    s['mean_eff_n'] = float((1/(W**2).sum(1)).mean())
    s['mean_max_abs_w'] = float(W.abs().max(1).mean())
    res[name]=s; store[name]=dict(series=series, wealth=wealth, W=W, pred=pred)

ridge = PooledLinearForecaster(ALPHA).fit(panel, tr); P_r = ridge.predict(panel, va)
ols = PooledLinearForecaster(0).fit(panel, tr);       P_o = ols.predict(panel, va)
hist = pd.DataFrame(np.tile(TARGETS.loc[tr, TICKERS].mean().to_numpy(), (len(va),1)), index=va, columns=TICKERS)
zero = pd.DataFrame(0.0, index=va, columns=TICKERS)
record('Ridge + MV (baseline)', P_r, GAMMA)
record('OLS + MV', P_o, OLS_GAMMA)
record('OLS + MV (gamma=5)', P_o, GAMMA)
record('Hist. mean + MV', hist, GAMMA)
record('Zero forecast = min-variance', zero, GAMMA)

bm = pd.read_csv('validation_benchmark_metrics.csv').set_index('strategy')
T = pd.DataFrame(res).T
T.to_csv('validation_results_person2.csv')
cols=['sharpe_net','gross_sharpe','ann_return','ann_volatility','max_drawdown','mean_weekly_turnover',
      'total_cost_fraction_sum','prediction_mse','r2_vs_zero','mean_rank_ic','mean_gross_leverage',
      'share_weeks_at_lev_cap','mean_short_exposure','mean_eff_n','solver_failures']
pd.set_option('display.width',250)
print(T[cols].astype(float).round(4).T)
print(bm[['sharpe_net','ann_return','ann_volatility','max_drawdown','mean_weekly_turnover']].round(4))
print('ridge coef (per 1 sd):', dict(zip(FEATURES, np.round(ridge.coef_*1e4,2))), 'b0bp', ridge.b0_*1e4)
print('ols coef (per 1 sd):', dict(zip(FEATURES, np.round(ols.coef_*1e4,2))))
print('pred cross-sec sd (bp): ridge', (P_r.std(1).mean()*1e4).round(2), 'ols', (P_o.std(1).mean()*1e4).round(2), 'actual', (act.std(1).mean()*1e4).round(1))

# cost sensitivity for baseline weights
W = store['Ridge + MV (baseline)']['W']
cs={}
for c in [0,2,5,10,20]:
    s,_,_ = evaluate_portfolio(W, act, cost_bps=c)
    cs[c]=s['sharpe_net']
print('cost sens', {k:round(v,3) for k,v in cs.items()})
# grid on validation: MSE vs Sharpe (diagnostic only, not selection)
grid=[]
for a in [0,100,1000,3000,10000,30000,100000]:
    P = PooledLinearForecaster(a).fit(panel,tr).predict(panel,va)
    for g in [1,2.5,5,10,25,50,100]:
        s,_,_,_ = run_strategy(P, TARGETS, PAST, va, g)
        grid.append(dict(alpha=a,gamma=g,mse=s['prediction_mse'],sharpe=s['sharpe_net'],turnover=s['mean_weekly_turnover']))
G=pd.DataFrame(grid); G.to_csv('validation_mse_vs_sharpe_grid.csv',index=False)
print(G.pivot(index='alpha',columns='gamma',values='sharpe').round(2))
print(G.groupby('alpha').mse.first().map(lambda v:f'{v:.8f}'))
import pickle; pickle.dump(dict(store=store, cs=cs, coef=dict(ridge=ridge.coef_, ols=ols.coef_)), open('store.pkl','wb'))


/tmp/ipykernel_2703/958167359.py:12: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  ic = [spearmanr(pred.loc[t], act.loc[t]).statistic for t in va]
/tmp/ipykernel_2703/958167359.py:13: RuntimeWarning: Mean of empty slice
  s['mean_rank_ic'] = float(np.nanmean(ic))


                         Ridge + MV (baseline)  OLS + MV  OLS + MV (gamma=5)  Hist. mean + MV  Zero forecast = min-variance
sharpe_net                              0.4371    0.6754              0.3971           1.4551                        0.8920
gross_sharpe                            0.5940    0.8668              0.5728           1.4653                        0.9141
ann_return                              0.0885    0.0506              0.0751           0.2520                        0.0334
ann_volatility                          0.3013    0.0776              0.2919           0.1640                        0.0376
max_drawdown                           -0.5033   -0.1256             -0.4821          -0.2132                       -0.0580
mean_weekly_turnover                    1.8223    0.5737              1.9754           0.0667                        0.0329
total_cost_fraction_sum                 0.1430    0.0450              0.1551           0.0052                        0.0026
predicti

## 4. Figures used in the slides

In [11]:
import pickle, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
d=pickle.load(open('store.pkl','rb')); S=d['store']
T=pd.read_csv('validation_results_person2.csv',index_col=0)
bm=pd.read_csv('validation_benchmark_metrics.csv').set_index('strategy')
G=pd.read_csv('validation_mse_vs_sharpe_grid.csv')
BLUE,ORANGE,AQUA,INK,MUTED,GRID='#2a78d6','#eb6834','#1baf7a','#0b0b0b','#52514e','#e4e3df'
plt.rcParams.update({'font.size':12,'axes.edgecolor':MUTED,'axes.labelcolor':INK,'xtick.color':MUTED,'ytick.color':MUTED,
  'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.color':GRID,'grid.linewidth':0.8,'axes.axisbelow':True,
  'figure.dpi':200,'savefig.bbox':'tight','legend.frameon':False})

# F1 MSE vs Sharpe
fig,ax=plt.subplots(figsize=(8,4.6))
ax.scatter(G.mse*1e4,G.sharpe,s=22,color='#b9b8b2',label='Ridge/OLS × risk-aversion grid (49 runs)',zorder=2)
pts={'Ridge + MV (baseline)':(BLUE,'Ridge + MV\n(our baseline)'),'Hist. mean + MV':(ORANGE,'Historical mean + MV'),
     'Zero forecast = min-variance':(AQUA,'Zero forecast\n(= min-variance)')}
for k,(c,l) in pts.items():
    r=T.loc[k]; ax.scatter(r.prediction_mse*1e4,r.sharpe_net,s=110,color=c,edgecolor='white',linewidth=2,zorder=4)
    ax.annotate(l,(r.prediction_mse*1e4,r.sharpe_net),xytext=(10,-4),textcoords='offset points',fontsize=11,color=INK,va='center')
for name,ls in [('SPY buy-and-hold','--'),('Weekly equal-weight',':')]:
    v=bm.loc[name,'sharpe_net']; ax.axhline(v,color=MUTED,ls=ls,lw=1.2)
    ax.text(10.07,v+0.03,f'{name} {v:.2f}',color=MUTED,fontsize=10,ha='right')
ax.set_xlabel('Forecast MSE on validation (×10⁻⁴)  ← more accurate'); ax.set_ylabel('Net Sharpe (validation)')
ax.set_xlim(9.62,10.1); ax.set_ylim(0,1.65); ax.legend(loc='lower left',fontsize=10)
fig.savefig('fig/f1_mse_vs_sharpe.png'); plt.close()

# F2 weights heatmap
W=S['Ridge + MV (baseline)']['W']
cmap=LinearSegmentedColormap.from_list('d',['#c2410c','#f3f2ee','#1d5fb0'])
fig,ax=plt.subplots(figsize=(10,5))
im=ax.imshow(W.T.values,aspect='auto',cmap=cmap,norm=TwoSlopeNorm(0,-0.5,1.0),interpolation='nearest')
ax.set_yticks(range(20),TICKERS,fontsize=9); ax.grid(False)
yrs=[i for i,t in enumerate(W.index) if i>0 and t.year!=W.index[i-1].year]
ax.set_xticks(yrs,[str(W.index[i].year) for i in yrs])
cb=fig.colorbar(im,ax=ax,fraction=0.03,pad=0.01); cb.set_label('Portfolio weight'); cb.outline.set_visible(False)
ax.set_title('Ridge + MV weights jump between corner positions week to week',loc='left',fontsize=14,color=INK)
fig.savefig('fig/f2_weights_heatmap.png'); plt.close()

# F3 wealth
fig,ax=plt.subplots(figsize=(8,4.6))
b=S['Ridge + MV (baseline)']; ser=b['series']
gross=pd.concat([pd.Series([1.0],index=[ser.index[0]-pd.Timedelta(days=7)]),(1+ser.gross_return).cumprod()])
ax.plot(gross.index,gross.values,color=BLUE,lw=1.5,ls='--',label='Ridge + MV, before costs')
ax.plot(b['wealth'].index,b['wealth'].values,color=BLUE,lw=2.2,label='Ridge + MV, net of 5 bp costs')
mv=S['Zero forecast = min-variance']['wealth']; ax.plot(mv.index,mv.values,color=AQUA,lw=2,label='Min-variance (no forecast)')
_,SP,TG,_=load_panels(); va=SP['validation']
for nm,alloc,ls in [('SPY buy-and-hold',{'SPY':1.0},'--'),('Weekly equal-weight',{t:0.05 for t in TICKERS},':')]:
    w=pd.DataFrame(0.0,index=va,columns=TICKERS)
    for k,v in alloc.items(): w[k]=v
    _,_,wl=evaluate_portfolio(w,TG.loc[va,TICKERS],allow_short=False,max_gross=1.0)
    ax.plot(wl.index,wl.values,color=MUTED,lw=1.4,ls=ls,label=nm)
import matplotlib.dates as mdates; ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y')); ax.set_ylabel('Wealth (starts at 1)'); ax.legend(loc='upper left',fontsize=10)
fig.savefig('fig/f3_wealth.png'); plt.close()

# F4 cost sensitivity + turnover
cs=d['cs']; fig,ax=plt.subplots(figsize=(6,4.2))
xs=list(cs.keys()); ys=list(cs.values())
ax.bar([str(x) for x in xs],ys,color=[BLUE if x!=5 else '#1d5fb0' for x in xs],width=0.6)
for x,y in zip(xs,ys): ax.text(str(x),(y+0.02) if y>=0 else 0.02,f'{y:.2f}',ha='center',fontsize=11,color=INK)
ax.set_ylim(-0.1,0.68); ax.axhline(0,color=MUTED,lw=1); ax.set_xlabel('Cost per dollar traded (bp)'); ax.set_ylabel('Net Sharpe, same weights')
ax.set_title('Cost sensitivity at 182% weekly turnover',loc='left',fontsize=13,color=INK); ax.grid(axis='x',visible=False)
fig.savefig('fig/f4_cost_sensitivity.png'); plt.close()
print('ok')


ok


## 5. Update the shared iteration log

In [12]:
# Append Person 2 rows to the shared iteration log
import csv, datetime as dt
today = dt.date.today().isoformat()
T = pd.read_csv('validation_results_person2.csv', index_col=0)
rows = [
 ['1a', today, 'Person 2', 'Forecast: pooled ridge (alpha=10000, chosen by fold MSE) + MV (gamma=5, chosen by fold Sharpe)',
  'An accuracy-trained forecast plus MV gives a usable portfolio', '2019-2021',
  round(T.loc['Ridge + MV (baseline)','sharpe_net'],4), f"{T.loc['Ridge + MV (baseline)','prediction_mse']:.6e}",
  round(T.loc['Ridge + MV (baseline)','mean_weekly_turnover'],4), 'Keep as baseline',
  'Below both benchmarks and min-variance; leverage cap binds every week; eff. N ~1.5; 14% cost drag'],
 ['1b', today, 'Person 2', 'OLS (alpha=0) + MV, gamma=100 (fold-best, grid edge)', 'Less shrinkage, more risk aversion', '2019-2021',
  round(T.loc['OLS + MV','sharpe_net'],4), f"{T.loc['OLS + MV','prediction_mse']:.6e}", round(T.loc['OLS + MV','mean_weekly_turnover'],4),
  'Reference', 'Higher Sharpe mainly via lower turnover (high gamma), not better forecasts'],
 ['1c', today, 'Person 2', 'Reference: training historical-mean forecast + MV (gamma=5)', 'Does forecast accuracy matter for the decision?', '2019-2021',
  round(T.loc['Hist. mean + MV','sharpe_net'],4), f"{T.loc['Hist. mean + MV','prediction_mse']:.6e}", round(T.loc['Hist. mean + MV','mean_weekly_turnover'],4),
  'Diagnostic only', 'Worse MSE, much better Sharpe; not pre-registered, partly a 2019-21 equity-market effect'],
 ['1d', today, 'Person 2', 'Reference: zero forecast = constrained min-variance', 'Value of any forecast', '2019-2021',
  round(T.loc['Zero forecast = min-variance','sharpe_net'],4), f"{T.loc['Zero forecast = min-variance','prediction_mse']:.6e}",
  round(T.loc['Zero forecast = min-variance','mean_weekly_turnover'],4), 'Diagnostic only', 'Beats the ridge baseline with no forecast at all'],
]
with open('iteration_log.csv','a',newline='') as f: csv.writer(f).writerows(rows)
print(pd.read_csv('iteration_log.csv').to_string())


     iteration      date                                                                                           owner                                              changed_component hypothesis  validation_period validation_sharpe_net validation_mse          turnover                                                                                           decision  notes
0          NaN      Team                                                                                        Protocol                      Freeze data/splits/metrics before fitting        NaN                NaN                                                                                                                                                   Pending    NaN
1a  2026-11-06  Person 2  Forecast: pooled ridge (alpha=10000, chosen by fold MSE) + MV (gamma=5, chosen by fold Sharpe)  An accuracy-trained forecast plus MV gives a usable portfolio  2019-2021             0.4371          9.716593e-04         1.8223

# 6. Experiment

Experiment 1: Distributionally robust optimisation

In [13]:
# EXPERIMENT: Robust portfolio optimisation
# Paper-inspired uncertainty penalty
# Uses the existing Ridge forecasts and backtester

from scipy.optimize import minimize
import numpy as np
import pandas as pd

def robust_mv_weights(mu, Sigma, gamma, rho=0.001,
                      allow_short=True):
    n = len(mu)
    scale = 1000.0

    bounds = (
        [(0, MAX_W)] * (2*n)
        if allow_short
        else [(0, MAX_W)] * n + [(0, 0)] * n
    )

    def objective(x):
        w = x[:n] - x[n:]
        utility = (
            mu @ w
            - 0.5 * gamma * w @ Sigma @ w
            - rho * np.linalg.norm(w)
        )
        return -scale * utility

    def gradient(x):
        w = x[:n] - x[n:]
        norm = max(np.linalg.norm(w), 1e-10)
        grad_w = -scale * (
            mu - gamma * Sigma @ w - rho*w/norm
        )
        return np.r_[grad_w, -grad_w]

    constraints = [
        {
            'type': 'eq',
            'fun': lambda x: x[:n].sum()-x[n:].sum()-1
        },
        {
            'type': 'ineq',
            'fun': lambda x: MAX_GROSS-x.sum()
        }
    ]

    x0 = np.r_[np.ones(n)/n, np.zeros(n)]

    result = minimize(
        objective, x0, jac=gradient,
        bounds=bounds,
        constraints=constraints,
        method='SLSQP',
        options={'maxiter':500, 'ftol':1e-10}
    )

    if not result.success:
        raise RuntimeError(result.message)

    w = result.x[:n]-result.x[n:]

    return w


def run_robust(rho):
    weights = []

    for date in va:
        Sigma = rolling_cov(PAST, date)
        mu = P_r.loc[date, TICKERS].to_numpy()

        w = robust_mv_weights(
            mu, Sigma, GAMMA, rho=rho
        )
        weights.append(w)

    W = pd.DataFrame(
        weights, index=va, columns=TICKERS
    )

    metrics, series, wealth = evaluate_portfolio(
        W,
        TARGETS.loc[va, TICKERS],
        predictions=P_r.loc[va, TICKERS],
        cost_bps=COST_BPS
    )

    return metrics, series, wealth, W


# Predefined sensitivity values
rho_values = [0, 0.0001, 0.0005, 0.001, 0.005]

rows = []
robust_outputs = {}

for rho in rho_values:
    metrics, series, wealth, weights = run_robust(rho)
    robust_outputs[rho] = (metrics, series, wealth, weights)

    rows.append({
        "rho": rho,
        "sharpe": metrics["sharpe_net"],
        "turnover": metrics["mean_weekly_turnover"],
        "drawdown": metrics["max_drawdown"],
        "annual_return": metrics["ann_return"]
    })

robust_results = pd.DataFrame(rows)

print(robust_results.round(4).to_string(index=False))

robust_results.to_csv(
    "robust_experiment_results.csv",
    index=False
)


   rho  sharpe  turnover  drawdown  annual_return
0.0000  0.4371    1.8223   -0.5033         0.0885
0.0001  0.4390    1.7910   -0.5027         0.0890
0.0005  0.4470    1.6619   -0.4984         0.0911
0.0010  0.4752    1.5499   -0.4876         0.0992
0.0050  0.6771    0.7737   -0.4025         0.1461


Experiment 2: Robust Optimisation + transaction cost penalty

In [14]:
# EXPERIMENT 2: Robust optimisation + transaction-cost penalty

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

RHO = 0.005
LAMBDAS = [0, 0.0001, 0.0005, 0.001, 0.002]

def cost_aware_weights(mu, Sigma, gamma, rho, lam, pretrade):
    n = len(mu)
    scale = 1000.0

    def objective(w):
        return -scale * (
            mu @ w
            - 0.5 * gamma * (w @ Sigma @ w)
            - rho * np.linalg.norm(w)
            - lam * np.sqrt((w - pretrade)**2 + 1e-12).sum()
        )

    def gradient(w):
        norm = max(np.linalg.norm(w), 1e-12)
        diff = w - pretrade

        return -scale * (
            mu - gamma * Sigma @ w
            - rho * w / norm
            - lam * diff / np.sqrt(diff**2 + 1e-12)
        )

    # Long-short, with gross leverage <= 1.5
    # Auxiliary variables u represent absolute positions.
    x0 = np.r_[np.ones(n)/n, np.ones(n)/n]

    def f(x):
        return objective(x[:n])

    def jac(x):
        return np.r_[gradient(x[:n]), np.zeros(n)]

    cons = [
        {'type':'eq',
         'fun':lambda x: x[:n].sum()-1},
        {'type':'ineq',
         'fun':lambda x: x[n:]-x[:n]},
        {'type':'ineq',
         'fun':lambda x: x[n:]+x[:n]},
        {'type':'ineq',
         'fun':lambda x: MAX_GROSS-x[n:].sum()}
    ]

    bounds = [(-MAX_W, MAX_W)]*n + [(0, MAX_W)]*n

    res = minimize(
        f, x0, jac=jac, bounds=bounds,
        constraints=cons, method='SLSQP',
        options={'maxiter':500, 'ftol':1e-10}
    )

    if not res.success:
        raise RuntimeError(res.message)

    w = res.x[:n]

    if (abs(w.sum()-1)>1e-6 or
        np.abs(w).sum()>MAX_GROSS+1e-6):
        raise ValueError("Constraint violation")

    return w


def run_cost_aware(lam):
    weights = []
    pretrade = np.zeros(len(TICKERS))

    for date in va:
        Sigma = rolling_cov(PAST, date)
        mu = P_r.loc[date, TICKERS].to_numpy()

        w = cost_aware_weights(
            mu, Sigma, GAMMA, RHO, lam, pretrade
        )

        weights.append(w)

        # Drift holdings after next-week realised returns
        r = TARGETS.loc[date, TICKERS].to_numpy()
        growth = 1 + np.dot(w, r)

        if growth <= 0:
            raise ValueError("Nonpositive portfolio growth")

        pretrade = w * (1+r) / growth

    W = pd.DataFrame(weights, index=va, columns=TICKERS)

    metrics, series, wealth = evaluate_portfolio(
        W, TARGETS.loc[va, TICKERS],
        predictions=P_r.loc[va, TICKERS],
        cost_bps=COST_BPS
    )

    return metrics, series, wealth, W


rows = []
cost_outputs = {}

for lam in LAMBDAS:
    m, s, wealth, W = run_cost_aware(lam)
    cost_outputs[lam] = (m, s, wealth, W)

    rows.append({
        'lambda': lam,
        'sharpe': m['sharpe_net'],
        'turnover': m['mean_weekly_turnover'],
        'drawdown': m['max_drawdown'],
        'annual_return': m['ann_return']
    })

cost_results = pd.DataFrame(rows)

print(cost_results.round(4).to_string(index=False))
cost_results.to_csv('cost_aware_results.csv', index=False)

# Visual comparison
plt.figure(figsize=(9, 5))
for lam in LAMBDAS:
    plt.plot(
        cost_outputs[lam][2].index,
        cost_outputs[lam][2].values,
        label=f'lambda={lam}'
    )

plt.title('Robust + transaction-cost-aware portfolios')
plt.ylabel('Cumulative net wealth')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('cost_aware_wealth.png', dpi=200, bbox_inches='tight')
plt.show()


 lambda  sharpe  turnover  drawdown  annual_return
 0.0000  0.6771    0.7737   -0.4025         0.1461
 0.0001  0.6931    0.6410   -0.3993         0.1498
 0.0005  0.7055    0.3252   -0.3972         0.1503
 0.0010  0.6276    0.1466   -0.3993         0.1256
 0.0020  0.7178    0.0462   -0.3597         0.1410


Experiment 3: end-to-end decision-aware learning

In [15]:

# EXPERIMENT 3: Decision-aware end-to-end learning
# Paper-inspired, simplified differentiable portfolio layer

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
torch.set_num_threads(2)

tr = SPLIT["train"]
va = SPLIT["validation"]
n = len(TICKERS)

# All preprocessing fitted on training data only
train_df = panel.loc[
    panel.index.get_level_values(0).isin(tr)
]
mean = train_df[FEATURES].mean().to_numpy(float)
std = train_df[FEATURES].std(ddof=0).to_numpy(float)
std = np.where(std < 1e-10, 1, std)

def make_data(dates):
    X, Y, C = [], [], []
    for date in dates:
        x = panel.loc[date].reindex(TICKERS)[FEATURES].to_numpy(float)
        X.append((x - mean) / std)
        Y.append(TARGETS.loc[date, TICKERS].to_numpy(float))
        C.append(rolling_cov(PAST, date))
    return (
        torch.tensor(np.array(X), dtype=torch.float32),
        torch.tensor(np.array(Y), dtype=torch.float32),
        torch.tensor(np.array(C), dtype=torch.float32)
    )


# Only use training dates with sufficient covariance history
# Preserve the original 52-week minimum

valid_train_dates = []

for date in tr:
    try:
        rolling_cov(PAST, date)
        valid_train_dates.append(date)
    except ValueError:
        continue

print("Original training weeks:", len(tr))
print("Usable training weeks:", len(valid_train_dates))
print("Excluded early weeks:", len(tr) - len(valid_train_dates))

# Rebuild training tensors using valid dates only
tr = pd.DatetimeIndex(valid_train_dates)

Xtr, Ytr, Ctr = make_data(tr)
Xva, Yva, Cva = make_data(va)

print("Training tensors:", Xtr.shape, Ytr.shape, Ctr.shape)
print("Validation tensors:", Xva.shape, Yva.shape, Cva.shape)

# A shared forecasting architecture
# Each ETF receives a forecast from the same linear model
class PortfolioModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(len(FEATURES), 1)

    def forward(self, X):
        return self.linear(X).squeeze(-1)

# Differentiable portfolio construction
# Long-only, fully invested
TEMPERATURE = 0.002
RISK_GAMMA = 5.0
COST = 0.0005

def portfolio_weights(pred):
    return torch.softmax(pred / TEMPERATURE, dim=1)

def decision_loss(model, X, Y, C):
    pred = model(X)
    w = portfolio_weights(pred)

    gross = (w * Y).sum(dim=1)
    risk = torch.einsum("bi,bij,bj->b", w, C, w)

    # Differentiable turnover approximation
    # First observation assumes deployment from cash
    previous = torch.cat(
        [torch.zeros_like(w[:1]), w[:-1]], dim=0
    )
    turnover = torch.sqrt(
        (w - previous)**2 + 1e-10
    ).sum(dim=1)

    utility = (
        gross
        - 0.5 * RISK_GAMMA * risk
        - COST * turnover
    )

    return -utility.mean()

# Internal holdout: last 26 training weeks
cut = len(tr) - 26

def fit_model(mode, epochs, full=False):
    model = PortfolioModel()

    # Start from zero forecasts -> equal weights
    with torch.no_grad():
        model.linear.weight.zero_()
        model.linear.bias.zero_()

    optimizer = torch.optim.Adam(
        model.parameters(), lr=0.0001
    )

    end = len(tr) if full else cut

    for epoch in range(epochs):
        optimizer.zero_grad()

        if mode == "MSE":
            loss = F.mse_loss(
                model(Xtr[:end]), Ytr[:end]
            )
        else:
            loss = decision_loss(
                model, Xtr[:end],
                Ytr[:end], Ctr[:end]
            )

        loss.backward()
        optimizer.step()

    return model

# Select training duration without touching 2019–2021
EPOCHS = [25, 50, 100, 200]
selection = []

for mode in ["MSE", "Decision-aware"]:
    for epochs in EPOCHS:
        model = fit_model(mode, epochs)
        with torch.no_grad():
            score = -decision_loss(
                model,
                Xtr[cut:], Ytr[cut:], Ctr[cut:]
            ).item()

        selection.append({
            "model": mode,
            "epochs": epochs,
            "holdout_utility": score
        })

selection_df = pd.DataFrame(selection)
print("INTERNAL TRAINING HOLDOUT")
print(selection_df.round(6).to_string(index=False))

# Refit on all 2010–2018 training observations
models = {}
for mode in ["MSE", "Decision-aware"]:
    subset = selection_df[
        selection_df["model"] == mode
    ]
    best_epochs = int(
        subset.loc[
            subset["holdout_utility"].idxmax(),
            "epochs"
        ]
    )
    models[mode] = fit_model(
        mode, best_epochs, full=True
    )
    print(mode, "selected epochs:", best_epochs)

# Evaluate with the ORIGINAL shared backtester
results = []
outputs = {}

for mode, model in models.items():
    with torch.no_grad():
        pred = model(Xva).numpy()
        weights = portfolio_weights(
            model(Xva)
        ).numpy()

    P = pd.DataFrame(
        pred, index=va, columns=TICKERS
    )
    W = pd.DataFrame(
        weights, index=va, columns=TICKERS
    )

    metrics, series, wealth = evaluate_portfolio(
        W,
        TARGETS.loc[va, TICKERS],
        predictions=P,
        cost_bps=COST_BPS,
        allow_short=False,
        max_gross=1.0
    )

    results.append({
        "model": mode,
        "sharpe": metrics["sharpe_net"],
        "mse": metrics["prediction_mse"],
        "turnover": metrics["mean_weekly_turnover"],
        "drawdown": metrics["max_drawdown"],
        "annual_return": metrics["ann_return"]
    })

    outputs[mode] = {
        "pred": P,
        "weights": W,
        "wealth": wealth,
        "metrics": metrics
    }

results3 = pd.DataFrame(results)
print("\nEXPERIMENT 3 RESULTS")
print(results3.round(6).to_string(index=False))

# Wealth comparison
plt.figure(figsize=(10, 5))
for mode in outputs:
    wealth = outputs[mode]["wealth"]
    plt.plot(
        wealth.index, wealth.values,
        label=mode, linewidth=2
    )

plt.title("MSE training vs decision-aware training")
plt.xlabel("Date")
plt.ylabel("Cumulative net wealth")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(
    "experiment3_end_to_end.png",
    dpi=200, bbox_inches="tight"
)
plt.show()

selection_df.to_csv(
    "experiment3_internal_selection.csv",
    index=False
)
results3.to_csv(
    "experiment3_results.csv",
    index=False
)


Original training weeks: 470
Usable training weeks: 419
Excluded early weeks: 51
Training tensors: torch.Size([419, 20, 9]) torch.Size([419, 20]) torch.Size([419, 20, 20])
Validation tensors: torch.Size([157, 20, 9]) torch.Size([157, 20]) torch.Size([157, 20, 20])
INTERNAL TRAINING HOLDOUT
         model  epochs  holdout_utility
           MSE      25        -0.003286
           MSE      50        -0.003046
           MSE     100        -0.003039
           MSE     200        -0.003043
Decision-aware      25        -0.002240
Decision-aware      50        -0.002462
Decision-aware     100        -0.002506
Decision-aware     200        -0.002527
MSE selected epochs: 100
Decision-aware selected epochs: 25

EXPERIMENT 3 RESULTS
         model   sharpe      mse  turnover  drawdown  annual_return
           MSE 0.880289 0.000971  0.455726 -0.372571       0.188794
Decision-aware 0.353819 0.001107  0.612817 -0.461261       0.055313


Experiment 4: proper rolling-fold hyperparameter tuning

In [16]:
# EXPERIMENT 4: Training-only rolling-fold tuning
# Run AFTER Experiments 1 and 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert "cost_aware_weights" in globals()
assert "folds" in globals()

# Fixed Ridge alpha and risk aversion from original baseline
TUNE_ALPHA = 10000
TUNE_GAMMA = 5.0

RHO_GRID = [0.0, 0.001, 0.005]
LAMBDA_GRID = [0.0, 0.0005, 0.002]

def backtest_robust(pred, dates, rho, lam):
    """Build sequential portfolio weights and evaluate
    using the original shared backtester."""

    dates = pd.DatetimeIndex(dates)
    weights = []
    pretrade = np.zeros(len(TICKERS))

    for date in dates:
        Sigma = rolling_cov(PAST, date)
        mu = pred.loc[date, TICKERS].to_numpy(float)

        w = cost_aware_weights(
            mu, Sigma, TUNE_GAMMA,
            rho, lam, pretrade
        )

        weights.append(w)

        # Drift portfolio using realised next-week returns.
        # These returns only affect the following decision.
        r = TARGETS.loc[date, TICKERS].to_numpy(float)
        growth = 1.0 + np.dot(w, r)

        if growth <= 0:
            raise ValueError("Invalid portfolio growth")

        pretrade = w * (1.0 + r) / growth

    W = pd.DataFrame(
        weights, index=dates, columns=TICKERS
    )

    m, series, wealth = evaluate_portfolio(
        W,
        TARGETS.loc[dates, TICKERS],
        predictions=pred.loc[dates, TICKERS],
        cost_bps=COST_BPS,
        allow_short=True,
        max_gross=MAX_GROSS,
        max_weight_abs=MAX_W
    )

    return m, series, wealth, W


# Precompute fold-specific forecasts
fold_predictions = []

for fold_id, (train_dates, val_dates) in enumerate(folds, 1):
    model = PooledLinearForecaster(
        TUNE_ALPHA
    ).fit(panel, train_dates)

    pred = model.predict(panel, val_dates)

    fold_predictions.append(
        (fold_id, val_dates, pred)
    )

# Tune both penalties
rows = []

for rho in RHO_GRID:
    for lam in LAMBDA_GRID:
        for fold_id, dates, pred in fold_predictions:

            m, series, wealth, W = backtest_robust(
                pred, dates, rho, lam
            )

            rows.append({
                "rho": rho,
                "lambda": lam,
                "fold": fold_id,
                "sharpe": m["sharpe_net"],
                "turnover": m["mean_weekly_turnover"],
                "drawdown": m["max_drawdown"],
                "annual_return": m["ann_return"]
            })

        print("Completed:", rho, lam)

fold_results = pd.DataFrame(rows)

# Aggregate the four folds
summary = (
    fold_results
    .groupby(["rho", "lambda"], as_index=False)
    .agg(
        mean_sharpe=("sharpe", "mean"),
        sd_sharpe=("sharpe", "std"),
        mean_turnover=("turnover", "mean"),
        mean_drawdown=("drawdown", "mean")
    )
)

summary = summary.sort_values(
    ["mean_sharpe", "mean_turnover"],
    ascending=[False, True]
)

print("\nTRAINING-ONLY TUNING RESULTS")
print(summary.round(4).to_string(index=False))

# Select parameters using training folds ONLY
best = summary.iloc[0]

BEST_RHO = float(best["rho"])
BEST_LAMBDA = float(best["lambda"])

print("\nSELECTED PARAMETERS")
print("rho =", BEST_RHO)
print("lambda =", BEST_LAMBDA)

fold_results.to_csv(
    "experiment4_fold_results.csv", index=False
)
summary.to_csv(
    "experiment4_tuning_summary.csv", index=False
)


Completed: 0.0 0.0
Completed: 0.0 0.0005
Completed: 0.0 0.002
Completed: 0.001 0.0
Completed: 0.001 0.0005
Completed: 0.001 0.002
Completed: 0.005 0.0
Completed: 0.005 0.0005
Completed: 0.005 0.002

TRAINING-ONLY TUNING RESULTS
  rho  lambda  mean_sharpe  sd_sharpe  mean_turnover  mean_drawdown
0.005  0.0000       1.7309     2.2766         0.1672        -0.0375
0.005  0.0005       1.6969     2.4218         0.0439        -0.0381
0.005  0.0020       1.6731     2.4233         0.0385        -0.0383
0.001  0.0000       1.4665     1.5262         0.6883        -0.0293
0.001  0.0005       1.2113     2.2436         0.0510        -0.0328
0.000  0.0000       1.2022     1.5664         1.4341        -0.0353
0.001  0.0020       1.0643     2.2751         0.0385        -0.0339
0.000  0.0005       0.9121     1.8377         0.0616        -0.0282
0.000  0.0020       0.7213     2.0973         0.0385        -0.0299

SELECTED PARAMETERS
rho = 0.005
lambda = 0.0


In [17]:
# EXPERIMENT 4: Validation evaluation
# No parameter selection from validation results

selected_metrics, selected_series, selected_wealth, selected_W = (
    backtest_robust(
        P_r, va,
        BEST_RHO, BEST_LAMBDA
    )
)

baseline_metrics = res["Ridge + MV (baseline)"]

comparison = pd.DataFrame({
    "Original Ridge": {
        "Net Sharpe": baseline_metrics["sharpe_net"],
        "Annual return": baseline_metrics["ann_return"],
        "Weekly turnover": baseline_metrics["mean_weekly_turnover"],
        "Max drawdown": baseline_metrics["max_drawdown"]
    },
    "Fold-selected robust model": {
        "Net Sharpe": selected_metrics["sharpe_net"],
        "Annual return": selected_metrics["ann_return"],
        "Weekly turnover": selected_metrics["mean_weekly_turnover"],
        "Max drawdown": selected_metrics["max_drawdown"]
    }
})

print("\nVALIDATION COMPARISON")
print(comparison.round(4).to_string())

comparison.to_csv(
    "experiment4_validation_comparison.csv"
)

plt.figure(figsize=(10, 5))

baseline_wealth = store[
    "Ridge + MV (baseline)"
]["wealth"]

plt.plot(
    baseline_wealth.index,
    baseline_wealth.values,
    label="Original Ridge",
    linewidth=2
)

plt.plot(
    selected_wealth.index,
    selected_wealth.values,
    label="Fold-selected robust model",
    linewidth=2
)

plt.title("Training-selected robust model vs Ridge baseline")
plt.xlabel("Date")
plt.ylabel("Cumulative net wealth")
plt.legend()
plt.grid(alpha=0.3)

plt.savefig(
    "experiment4_validation.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()



VALIDATION COMPARISON
                 Original Ridge  Fold-selected robust model
Net Sharpe               0.4371                      0.6771
Annual return            0.0885                      0.1461
Weekly turnover          1.8223                      0.7737
Max drawdown            -0.5033                     -0.4025


In [18]:

print("SELECTED PARAMETERS")
print("rho:", BEST_RHO)
print("lambda:", BEST_LAMBDA)

print("\nTOP TRAINING-FOLD CONFIGURATIONS")
print(summary.head(9).round(4).to_string(index=False))

print("\nPER-FOLD RESULTS FOR SELECTED MODEL")
selected_folds = fold_results[
    np.isclose(fold_results["rho"], BEST_RHO) &
    np.isclose(fold_results["lambda"], BEST_LAMBDA)
]
print(selected_folds.round(4).to_string(index=False))


SELECTED PARAMETERS
rho: 0.005
lambda: 0.0

TOP TRAINING-FOLD CONFIGURATIONS
  rho  lambda  mean_sharpe  sd_sharpe  mean_turnover  mean_drawdown
0.005  0.0000       1.7309     2.2766         0.1672        -0.0375
0.005  0.0005       1.6969     2.4218         0.0439        -0.0381
0.005  0.0020       1.6731     2.4233         0.0385        -0.0383
0.001  0.0000       1.4665     1.5262         0.6883        -0.0293
0.001  0.0005       1.2113     2.2436         0.0510        -0.0328
0.000  0.0000       1.2022     1.5664         1.4341        -0.0353
0.001  0.0020       1.0643     2.2751         0.0385        -0.0339
0.000  0.0005       0.9121     1.8377         0.0616        -0.0282
0.000  0.0020       0.7213     2.0973         0.0385        -0.0299

PER-FOLD RESULTS FOR SELECTED MODEL
  rho  lambda  fold  sharpe  turnover  drawdown  annual_return
0.005     0.0     1  4.1206    0.1531   -0.0138         0.1775
0.005     0.0     2  3.1609    0.1400   -0.0117         0.1312
0.005     0.0    

In [19]:

print("SELECTED PARAMETERS")
print("rho:", BEST_RHO)
print("lambda:", BEST_LAMBDA)

print("\nTOP TRAINING-FOLD CONFIGURATIONS")
print(summary.head(9).round(4).to_string(index=False))

print("\nPER-FOLD RESULTS FOR SELECTED MODEL")
selected_folds = fold_results[
    np.isclose(fold_results["rho"], BEST_RHO) &
    np.isclose(fold_results["lambda"], BEST_LAMBDA)
]
print(selected_folds.round(4).to_string(index=False))


SELECTED PARAMETERS
rho: 0.005
lambda: 0.0

TOP TRAINING-FOLD CONFIGURATIONS
  rho  lambda  mean_sharpe  sd_sharpe  mean_turnover  mean_drawdown
0.005  0.0000       1.7309     2.2766         0.1672        -0.0375
0.005  0.0005       1.6969     2.4218         0.0439        -0.0381
0.005  0.0020       1.6731     2.4233         0.0385        -0.0383
0.001  0.0000       1.4665     1.5262         0.6883        -0.0293
0.001  0.0005       1.2113     2.2436         0.0510        -0.0328
0.000  0.0000       1.2022     1.5664         1.4341        -0.0353
0.001  0.0020       1.0643     2.2751         0.0385        -0.0339
0.000  0.0005       0.9121     1.8377         0.0616        -0.0282
0.000  0.0020       0.7213     2.0973         0.0385        -0.0299

PER-FOLD RESULTS FOR SELECTED MODEL
  rho  lambda  fold  sharpe  turnover  drawdown  annual_return
0.005     0.0     1  4.1206    0.1531   -0.0138         0.1775
0.005     0.0     2  3.1609    0.1400   -0.0117         0.1312
0.005     0.0    

Experimen 5: Ablation Study

In [20]:
# EXPERIMENT 5: Ablation study
# Uses existing backtest_robust() from Experiment 4

import pandas as pd
import matplotlib.pyplot as plt

ABLATIONS = {
    "Baseline": (0.0, 0.0),
    "Robust only": (BEST_RHO, 0.0),
    "Turnover only": (0.0, 0.002),
    "Both penalties": (BEST_RHO, 0.002)
}

rows = []
ablation_outputs = {}

for name, (rho, lam) in ABLATIONS.items():

    metrics, series, wealth, weights = backtest_robust(
        P_r, va, rho, lam
    )

    rows.append({
        "model": name,
        "rho": rho,
        "lambda": lam,
        "sharpe": metrics["sharpe_net"],
        "annual_return": metrics["ann_return"],
        "turnover": metrics["mean_weekly_turnover"],
        "drawdown": metrics["max_drawdown"],
        "volatility": metrics["ann_volatility"]
    })

    ablation_outputs[name] = {
        "metrics": metrics,
        "series": series,
        "wealth": wealth,
        "weights": weights
    }

ablation_results = pd.DataFrame(rows)

print("EXPERIMENT 5: ABLATION RESULTS")
print(ablation_results.round(4).to_string(index=False))

ablation_results.to_csv(
    "experiment5_ablation.csv",
    index=False
)

# Plot cumulative wealth
plt.figure(figsize=(10, 5))

for name, result in ablation_outputs.items():
    wealth = result["wealth"]

    plt.plot(
        wealth.index,
        wealth.values,
        label=name,
        linewidth=2
    )

plt.title("Ablation study: robust and turnover penalties")
plt.xlabel("Date")
plt.ylabel("Cumulative net wealth")
plt.legend()
plt.grid(alpha=0.3)

plt.savefig(
    "experiment5_ablation_wealth.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


EXPERIMENT 5: ABLATION RESULTS
         model   rho  lambda  sharpe  annual_return  turnover  drawdown  volatility
      Baseline 0.000   0.000  0.4371         0.0885    1.8223   -0.5033      0.3013
   Robust only 0.005   0.000  0.6771         0.1461    0.7737   -0.4025      0.2474
 Turnover only 0.000   0.002  0.1458        -0.0021    0.0792   -0.5221      0.2875
Both penalties 0.005   0.002  0.7178         0.1410    0.0462   -0.3597      0.2174


In [21]:

import os

print("Available CSV files:")
for filename in sorted(os.listdir(".")):
    if filename.endswith(".csv"):
        print(" -", filename)

print("\nSelected final model:")
print("rho =", BEST_RHO)
print("lambda =", BEST_LAMBDA)


Available CSV files:
 - cost_aware_results.csv
 - experiment3_internal_selection.csv
 - experiment3_results.csv
 - experiment4_fold_results.csv
 - experiment4_tuning_summary.csv
 - experiment4_validation_comparison.csv
 - experiment5_ablation.csv
 - iteration_log.csv
 - robust_experiment_results.csv
 - train_panel.csv
 - training_arch_lm_tests.csv
 - training_correlations.csv
 - tuning_folds.csv
 - validation_benchmark_metrics.csv
 - validation_mse_vs_sharpe_grid.csv
 - validation_panel.csv
 - validation_results_person2.csv
 - walk_forward_fold_schedule.csv

Selected final model:
rho = 0.005
lambda = 0.0


Transaction-cost stress test

In [22]:

# EXPERIMENT 6A: TRANSACTION COST STRESS TEST

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

COST_LEVELS = [0, 2, 5, 10, 20, 30, 50]

# Reuse the already calculated portfolios.
# No new optimisation or parameter selection.
baseline_W = ablation_outputs["Baseline"]["weights"]
robust_W = ablation_outputs["Robust only"]["weights"]
combined_W = ablation_outputs["Both penalties"]["weights"]

PORTFOLIOS = {
    "Original Ridge": baseline_W,
    "Robust only": robust_W,
    "Robust + turnover": combined_W
}

cost_rows = []

for name, W in PORTFOLIOS.items():
    for cost in COST_LEVELS:

        metrics, series, wealth = evaluate_portfolio(
            W,
            TARGETS.loc[va, TICKERS],
            predictions=P_r.loc[va, TICKERS],
            cost_bps=cost,
            allow_short=True,
            max_gross=MAX_GROSS,
            max_weight_abs=MAX_W
        )

        cost_rows.append({
            "model": name,
            "cost_bps": cost,
            "net_sharpe": metrics["sharpe_net"],
            "annual_return": metrics["ann_return"],
            "max_drawdown": metrics["max_drawdown"],
            "turnover": metrics["mean_weekly_turnover"]
        })

cost_stress = pd.DataFrame(cost_rows)

print("TRANSACTION COST STRESS TEST")
print(
    cost_stress.pivot(
        index="cost_bps",
        columns="model",
        values="net_sharpe"
    ).round(3).to_string()
)

plt.figure(figsize=(10, 5))

for name, group in cost_stress.groupby("model"):
    plt.plot(
        group["cost_bps"],
        group["net_sharpe"],
        marker="o",
        label=name
    )

plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Transaction cost (basis points)")
plt.ylabel("Net Sharpe")
plt.title("Sensitivity of portfolio performance to trading costs")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("experiment6_cost_stress.png", dpi=200)
plt.show()

cost_stress.to_csv(
    "experiment6_cost_stress.csv",
    index=False
)


TRANSACTION COST STRESS TEST
model     Original Ridge  Robust + turnover  Robust only
cost_bps                                                
0                  0.594              0.723        0.758
2                  0.531              0.721        0.725
5                  0.437              0.718        0.677
10                 0.280              0.712        0.596
20                -0.038              0.701        0.434
30                -0.357              0.690        0.270
50                -1.004              0.669       -0.059


Moving-block bootstrap

In [23]:

# EXPERIMENT 7: MOVING-BLOCK BOOTSTRAP
# Compare net Sharpe of robust vs baseline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

base_returns = ablation_outputs["Baseline"]["series"]["net_return"]
robust_returns = ablation_outputs["Robust only"]["series"]["net_return"]

paired = pd.concat(
    [base_returns.rename("baseline"),
     robust_returns.rename("robust")],
    axis=1
).dropna()

def sharpe(r):
    sd = np.std(r, ddof=1)
    if sd < 1e-12:
        return np.nan
    return np.mean(r) / sd * np.sqrt(52)

observed = (
    sharpe(paired["robust"].values)
    - sharpe(paired["baseline"].values)
)

rng = np.random.default_rng(42)

B = 2000
BLOCK = 8
N = len(paired)

bootstrap_diffs = []

for _ in range(B):

    indices = []

    while len(indices) < N:
        start = rng.integers(0, N - BLOCK + 1)
        indices.extend(range(start, start + BLOCK))

    indices = indices[:N]

    sample = paired.iloc[indices]

    diff = (
        sharpe(sample["robust"].values)
        - sharpe(sample["baseline"].values)
    )

    bootstrap_diffs.append(diff)

bootstrap_diffs = np.array(bootstrap_diffs)

ci_low, ci_high = np.nanpercentile(
    bootstrap_diffs, [2.5, 97.5]
)

print("OBSERVED SHARPE IMPROVEMENT:", round(observed, 4))
print("95% BOOTSTRAP CI:", round(ci_low, 4), round(ci_high, 4))

if ci_low > 0:
    print("CI excludes zero: evidence of a positive Sharpe difference.")
else:
    print("CI includes zero: improvement is statistically uncertain.")

plt.figure(figsize=(9, 5))
plt.hist(bootstrap_diffs, bins=40, alpha=0.75)
plt.axvline(0, color="black", linestyle="--", label="No improvement")
plt.axvline(observed, color="red", label="Observed difference")
plt.xlabel("Robust Sharpe minus baseline Sharpe")
plt.ylabel("Bootstrap frequency")
plt.title("Moving-block bootstrap: Sharpe improvement")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("experiment7_bootstrap.png", dpi=200)
plt.show()

pd.DataFrame({
    "sharpe_difference": bootstrap_diffs
}).to_csv("experiment7_bootstrap.csv", index=False)


OBSERVED SHARPE IMPROVEMENT: 0.2399
95% BOOTSTRAP CI: -0.0019 1.0451
CI includes zero: improvement is statistically uncertain.


Portfolio concentraiton and stability

In [24]:

# EXPERIMENT 8: PORTFOLIO STABILITY AND CONCENTRATION

stability_rows = []

for name, result in ablation_outputs.items():

    W = result["weights"].copy()

    gross = W.abs().sum(axis=1)

    # Concentration based on absolute exposure shares
    exposure_share = W.abs().div(gross, axis=0)
    hhi = (exposure_share ** 2).sum(axis=1)

    max_weight = W.abs().max(axis=1)

    short_exposure = (
        -W.clip(upper=0)
    ).sum(axis=1)

    weight_changes = W.diff().abs().sum(axis=1)

    stability_rows.append({
        "model": name,
        "mean_HHI": hhi.mean(),
        "effective_assets": (1 / hhi).mean(),
        "max_abs_weight": max_weight.mean(),
        "gross_leverage": gross.mean(),
        "short_exposure": short_exposure.mean(),
        "weight_change": weight_changes.iloc[1:].mean()
    })

stability_results = pd.DataFrame(stability_rows)

print("PORTFOLIO STABILITY RESULTS")
print(stability_results.round(4).to_string(index=False))

stability_results.to_csv(
    "experiment8_portfolio_stability.csv",
    index=False
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(
    stability_results["model"],
    stability_results["effective_assets"]
)
axes[0].set_title("Effective number of assets")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(
    stability_results["model"],
    stability_results["weight_change"]
)
axes[1].set_title("Average weekly change in target weights")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("experiment8_stability.png", dpi=200)
plt.show()


PORTFOLIO STABILITY RESULTS
         model  mean_HHI  effective_assets  max_abs_weight  gross_leverage  short_exposure  weight_change
      Baseline    0.3245            3.2944          0.6821          1.5000          0.2500         1.8136
   Robust only    0.0817           12.8879          0.1851          1.2533          0.1266         0.7618
 Turnover only    0.3154            4.0259          0.5470          1.3581          0.1790         0.0866
Both penalties    0.0766           13.5568          0.1597          1.0886          0.0443         0.0504


Experiment 9: Improved End-to-End robust learning

In [25]:

# EXPERIMENT 9A: Differentiable robust decision loss

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

RISK_GAMMA = 5.0
RHO_E2E = 0.005
LAMBDA_E2E = 0.0005
COST_E2E = COST_BPS / 10000

def improved_decision_loss(
    model, X, Y, C,
    rho=RHO_E2E,
    lam=LAMBDA_E2E
):
    pred = model(X)
    w = portfolio_weights(pred)

    # Realised gross return
    gross_return = (w * Y).sum(dim=1)

    # Portfolio variance
    risk = torch.einsum(
        "bi,bij,bj->b", w, C, w
    )

    # Norm-based uncertainty penalty
    uncertainty = torch.linalg.vector_norm(
        w, dim=1
    )

    # Holdings before each rebalance
    # The previous target weights drift with realised returns.
    previous_w = w[:-1]
    previous_r = Y[:-1]

    growth = 1.0 + (
        previous_w * previous_r
    ).sum(dim=1)

    drifted = (
        previous_w * (1.0 + previous_r)
        / growth.unsqueeze(1).clamp(min=1e-8)
    )

    # First week: deployment from cash
    initial = torch.zeros_like(w[:1])

    pretrade = torch.cat(
        [initial, drifted], dim=0
    )

    # Smooth absolute turnover penalty
    turnover = torch.sqrt(
        (w - pretrade)**2 + 1e-10
    ).sum(dim=1)

    utility = (
        gross_return
        - 0.5 * RISK_GAMMA * risk
        - rho * uncertainty
        - lam * turnover
        - COST_E2E * turnover
    )

    return -utility.mean()


In [26]:

# EXPERIMENT 9B: Train and select models
# All model selection uses training data only

SEEDS = [42, 123, 456]
EPOCH_GRID = [25, 50, 100, 200]

# The Experiment 3 training split
cut = len(Xtr) - 26

def fit_e2e(mode, epochs, seed, full=False):
    torch.manual_seed(seed)

    model = PortfolioModel()

    with torch.no_grad():
        model.linear.weight.zero_()
        model.linear.bias.zero_()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.0001
    )

    end = len(Xtr) if full else cut

    for _ in range(epochs):
        optimizer.zero_grad()

        if mode == "MSE":
            loss = F.mse_loss(
                model(Xtr[:end]), Ytr[:end]
            )

        elif mode == "Decision-aware":
            loss = improved_decision_loss(
                model,
                Xtr[:end],
                Ytr[:end],
                Ctr[:end],
                rho=0,
                lam=LAMBDA_E2E
            )

        elif mode == "Robust decision-aware":
            loss = improved_decision_loss(
                model,
                Xtr[:end],
                Ytr[:end],
                Ctr[:end],
                rho=RHO_E2E,
                lam=LAMBDA_E2E
            )

        else:
            raise ValueError(mode)

        loss.backward()
        optimizer.step()

    return model


MODES = [
    "MSE",
    "Decision-aware",
    "Robust decision-aware"
]

selection_rows = []

for mode in MODES:
    for seed in SEEDS:
        for epochs in EPOCH_GRID:

            model = fit_e2e(
                mode, epochs, seed, full=False
            )

            with torch.no_grad():
                score = -improved_decision_loss(
                    model,
                    Xtr[cut:],
                    Ytr[cut:],
                    Ctr[cut:],
                    rho=RHO_E2E,
                    lam=LAMBDA_E2E
                ).item()

            selection_rows.append({
                "model": mode,
                "seed": seed,
                "epochs": epochs,
                "holdout_utility": score
            })

selection9 = pd.DataFrame(selection_rows)

print("TRAINING HOLDOUT RESULTS")
print(selection9.round(6).to_string(index=False))

# Select epochs using average holdout utility
epoch_summary = (
    selection9
    .groupby(["model", "epochs"], as_index=False)
    ["holdout_utility"].mean()
)

best_epochs = {}

for mode in MODES:
    subset = epoch_summary[
        epoch_summary["model"] == mode
    ]

    best_row = subset.loc[
        subset["holdout_utility"].idxmax()
    ]

    best_epochs[mode] = int(
        best_row["epochs"]
    )

    print(
        mode,
        "selected epochs:",
        best_epochs[mode]
    )

selection9.to_csv(
    "experiment9_training_selection.csv",
    index=False
)


TRAINING HOLDOUT RESULTS
                model  seed  epochs  holdout_utility
                  MSE    42      25        -0.004609
                  MSE    42      50        -0.004411
                  MSE    42     100        -0.004415
                  MSE    42     200        -0.004418
                  MSE   123      25        -0.004609
                  MSE   123      50        -0.004411
                  MSE   123     100        -0.004415
                  MSE   123     200        -0.004418
                  MSE   456      25        -0.004609
                  MSE   456      50        -0.004411
                  MSE   456     100        -0.004415
                  MSE   456     200        -0.004418
       Decision-aware    42      25        -0.004661
       Decision-aware    42      50        -0.004981
       Decision-aware    42     100        -0.005164
       Decision-aware    42     200        -0.005178
       Decision-aware   123      25        -0.004661
       Decision-aware

In [27]:

# EXPERIMENT 9C: Validation comparison

results9 = []
outputs9 = {}

for mode in MODES:
    model = fit_e2e(
        mode,
        best_epochs[mode],
        seed=42,
        full=True
    )

    with torch.no_grad():
        pred = model(Xva)
        weights = portfolio_weights(pred)

    P = pd.DataFrame(
        pred.numpy(),
        index=va,
        columns=TICKERS
    )

    W = pd.DataFrame(
        weights.numpy(),
        index=va,
        columns=TICKERS
    )

    assert np.all(W.to_numpy() >= -1e-8)
    assert np.allclose(W.sum(axis=1), 1, atol=1e-6)

    metrics, series, wealth = evaluate_portfolio(
        W,
        TARGETS.loc[va, TICKERS],
        predictions=P,
        cost_bps=COST_BPS,
        allow_short=False,
        max_gross=1.0
    )

    results9.append({
        "model": mode,
        "sharpe": metrics["sharpe_net"],
        "mse": metrics["prediction_mse"],
        "turnover": metrics["mean_weekly_turnover"],
        "annual_return": metrics["ann_return"],
        "drawdown": metrics["max_drawdown"]
    })

    outputs9[mode] = {
        "model": model,
        "predictions": P,
        "weights": W,
        "series": series,
        "wealth": wealth,
        "metrics": metrics
    }

results9 = pd.DataFrame(results9)

print("EXPERIMENT 9 RESULTS")
print(results9.round(6).to_string(index=False))

results9.to_csv(
    "experiment9_results.csv",
    index=False
)

plt.figure(figsize=(10, 5))

for mode, output in outputs9.items():
    wealth = output["wealth"]

    plt.plot(
        wealth.index,
        wealth.values,
        label=mode,
        linewidth=2
    )

plt.title("Improved end-to-end learning: validation wealth")
plt.xlabel("Date")
plt.ylabel("Cumulative net wealth")
plt.legend()
plt.grid(alpha=0.3)

plt.savefig(
    "experiment9_wealth.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


EXPERIMENT 9 RESULTS
                model   sharpe      mse  turnover  annual_return  drawdown
                  MSE 0.886193 0.000971  0.441671       0.189286 -0.367120
       Decision-aware 0.385506 0.001115  0.418784       0.060837 -0.419853
Robust decision-aware 0.959333 0.001048  0.114748       0.152542 -0.285360


## Step 1 - Train all Four Models

In [28]:

# EXPERIMENT 10A: Four-way end-to-end ablation
# Run after Experiment 9

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

ABLATION10 = {
    "A: Standard": (0.0, 0.0),
    "B: Uncertainty only": (RHO_E2E, 0.0),
    "C: Turnover only": (0.0, LAMBDA_E2E),
    "D: Both penalties": (RHO_E2E, LAMBDA_E2E)
}

FIXED_EPOCHS = 25
LEARNING_RATE = 0.0001

def fit_ablation10(rho, lam):
    torch.manual_seed(42)

    model = PortfolioModel()

    with torch.no_grad():
        model.linear.weight.zero_()
        model.linear.bias.zero_()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    history = []

    for epoch in range(FIXED_EPOCHS):
        optimizer.zero_grad()

        loss = improved_decision_loss(
            model, Xtr, Ytr, Ctr,
            rho=rho,
            lam=lam
        )

        loss.backward()
        optimizer.step()

        history.append(float(loss.item()))

    return model, history


models10 = {}
histories10 = {}

for name, (rho, lam) in ABLATION10.items():
    model, history = fit_ablation10(rho, lam)
    models10[name] = model
    histories10[name] = history

    print(
        name,
        "| final training loss:",
        round(history[-1], 6)
    )

plt.figure(figsize=(9, 5))

for name, history in histories10.items():
    plt.plot(history, label=name)

plt.xlabel("Training epoch")
plt.ylabel("Negative portfolio utility")
plt.title("End-to-end training convergence")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("experiment10_training.png", dpi=200)
plt.show()


A: Standard | final training loss: -0.001594
B: Uncertainty only | final training loss: -8.4e-05
C: Turnover only | final training loss: -0.001314
D: Both penalties | final training loss: 4e-06


## Step 2 - Evaluate all four models

In [29]:

# EXPERIMENT 10B: Validation comparison

results10 = []
outputs10 = {}

for name, model in models10.items():

    model.eval()

    with torch.no_grad():
        pred = model(Xva)
        weights = portfolio_weights(pred)

    P = pd.DataFrame(
        pred.numpy(),
        index=va,
        columns=TICKERS
    )

    W = pd.DataFrame(
        weights.numpy(),
        index=va,
        columns=TICKERS
    )

    assert np.all(W.to_numpy() >= -1e-8)
    assert np.allclose(W.sum(axis=1), 1, atol=1e-6)

    metrics, series, wealth = evaluate_portfolio(
        W,
        TARGETS.loc[va, TICKERS],
        predictions=P,
        cost_bps=COST_BPS,
        allow_short=False,
        max_gross=1.0
    )

    results10.append({
        "model": name,
        "rho": ABLATION10[name][0],
        "lambda": ABLATION10[name][1],
        "sharpe": metrics["sharpe_net"],
        "mse": metrics["prediction_mse"],
        "turnover": metrics["mean_weekly_turnover"],
        "annual_return": metrics["ann_return"],
        "drawdown": metrics["max_drawdown"]
    })

    outputs10[name] = {
        "predictions": P,
        "weights": W,
        "series": series,
        "wealth": wealth,
        "metrics": metrics
    }

results10 = pd.DataFrame(results10)

print("\nEXPERIMENT 10: ABLATION RESULTS")
print(results10.round(6).to_string(index=False))

results10.to_csv(
    "experiment10_ablation.csv",
    index=False
)

plt.figure(figsize=(10, 5))

for name, output in outputs10.items():
    wealth = output["wealth"]

    plt.plot(
        wealth.index,
        wealth.values,
        label=name,
        linewidth=2
    )

plt.title("End-to-end ablation: validation wealth")
plt.xlabel("Date")
plt.ylabel("Cumulative net wealth")
plt.legend()
plt.grid(alpha=0.3)

plt.savefig(
    "experiment10_wealth.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()



EXPERIMENT 10: ABLATION RESULTS
              model   rho  lambda   sharpe      mse  turnover  annual_return  drawdown
        A: Standard 0.000  0.0000 0.341625 0.001105  0.613216       0.052249 -0.461108
B: Uncertainty only 0.005  0.0000 0.858297 0.001002  0.288975       0.151332 -0.331291
   C: Turnover only 0.000  0.0005 0.385506 0.001115  0.418784       0.060837 -0.419853
  D: Both penalties 0.005  0.0005 0.959333 0.001048  0.114748       0.152542 -0.285360


## Step 3 - Quantify the contribution of each penalty

In [30]:

# EXPERIMENT 10C: Factorial effects

scores = results10.set_index("model")["sharpe"]

A = scores["A: Standard"]
B = scores["B: Uncertainty only"]
C = scores["C: Turnover only"]
D = scores["D: Both penalties"]

effects = pd.Series({
    "Uncertainty effect without turnover": B - A,
    "Turnover effect without uncertainty": C - A,
    "Uncertainty effect with turnover": D - C,
    "Turnover effect with uncertainty": D - B,
    "Interaction effect": D - B - C + A
})

print("EXPERIMENT 10: FACTORIAL EFFECTS")
print(effects.round(4).to_string())

effects.to_csv("experiment10_effects.csv")


EXPERIMENT 10: FACTORIAL EFFECTS
Uncertainty effect without turnover    0.5167
Turnover effect without uncertainty    0.0439
Uncertainty effect with turnover       0.5738
Turnover effect with uncertainty       0.1010
Interaction effect                     0.0572


# Auditing the Data

In [31]:

# EXPERIMENT 11A: DATA INTEGRITY AUDIT

import numpy as np
import pandas as pd

train_dates = pd.DatetimeIndex(SPLIT["train"])
validation_dates = pd.DatetimeIndex(SPLIT["validation"])

print("=== DATA INTEGRITY AUDIT ===")

print("\n1. DATASET PERIODS")
print("Training:", train_dates.min(), "to", train_dates.max())
print("Validation:", validation_dates.min(), "to", validation_dates.max())

assert train_dates.max() < validation_dates.min(), (
    "FAIL: Training and validation overlap!"
)
assert train_dates.is_unique
assert validation_dates.is_unique

print("PASS: Training and validation dates are separated.")

print("\n2. PANEL COMPLETENESS")

for label, dates in [
    ("Training", train_dates),
    ("Validation", validation_dates)
]:
    sample = panel.loc[
        panel.index.get_level_values(0).isin(dates)
    ]

    expected = len(dates) * len(TICKERS)
    actual = len(sample)

    print(label, "rows:", actual, "expected:", expected)

    assert actual == expected, (
        f"FAIL: Missing or duplicate {label} observations"
    )

    assert np.isfinite(
        sample[FEATURES].to_numpy(dtype=float)
    ).all()

print("PASS: Panel dimensions and features are valid.")

print("\n3. TARGET COMPLETENESS")

for label, dates in [
    ("Training", train_dates),
    ("Validation", validation_dates)
]:
    y = TARGETS.loc[dates, TICKERS]

    assert y.notna().all().all()
    assert np.isfinite(y.to_numpy(dtype=float)).all()

print("PASS: All target returns are finite.")

print("\n4. COVARIANCE HISTORY")

valid_cov_dates = []
invalid_cov_dates = []

for date in train_dates:
    try:
        Sigma = rolling_cov(PAST, date)

        assert Sigma.shape == (
            len(TICKERS), len(TICKERS)
        )
        assert np.isfinite(Sigma).all()
        assert np.allclose(Sigma, Sigma.T, atol=1e-8)

        valid_cov_dates.append(date)

    except ValueError:
        invalid_cov_dates.append(date)

print("Training weeks:", len(train_dates))
print("Usable covariance weeks:", len(valid_cov_dates))
print("Excluded weeks:", len(invalid_cov_dates))

assert len(valid_cov_dates) == len(Xtr), (
    "Mismatch between covariance history and E2E training tensors"
)

print("\nDATA INTEGRITY CHECKS COMPLETED")


=== DATA INTEGRITY AUDIT ===

1. DATASET PERIODS
Training: 2009-12-25 00:00:00 to 2018-12-21 00:00:00
Validation: 2018-12-28 00:00:00 to 2021-12-24 00:00:00
PASS: Training and validation dates are separated.

2. PANEL COMPLETENESS
Training rows: 9400 expected: 9400
Validation rows: 3140 expected: 3140
PASS: Panel dimensions and features are valid.

3. TARGET COMPLETENESS
PASS: All target returns are finite.

4. COVARIANCE HISTORY
Training weeks: 470
Usable covariance weeks: 419
Excluded weeks: 51

DATA INTEGRITY CHECKS COMPLETED


# Verifying portfolio constraints and accounting

In [32]:

# EXPERIMENT 11B: PORTFOLIO AND BACKTEST AUDIT

import numpy as np
import pandas as pd

print("=== PORTFOLIO AUDIT ===")

audit_rows = []

# End-to-end models are long-only
for name, output in outputs10.items():

    W = output["weights"]
    r = output["series"]["net_return"].dropna()

    min_weight = W.min().min()
    max_sum_error = (W.sum(axis=1) - 1).abs().max()

    assert min_weight >= -1e-7
    assert max_sum_error < 1e-5
    assert np.isfinite(r).all()
    assert (r > -1).all()

    calculated_sharpe = (
        r.mean() / r.std(ddof=1) * np.sqrt(52)
    )

    reported_sharpe = output["metrics"]["sharpe_net"]

    audit_rows.append({
        "model": name,
        "min_weight": min_weight,
        "max_budget_error": max_sum_error,
        "calculated_sharpe": calculated_sharpe,
        "reported_sharpe": reported_sharpe,
        "sharpe_difference": (
            calculated_sharpe - reported_sharpe
        )
    })

audit = pd.DataFrame(audit_rows)

print(audit.round(8).to_string(index=False))

assert audit["sharpe_difference"].abs().max() < 1e-5, (
    "Sharpe calculation differs from original backtester"
)

print("\nPASS: E2E portfolio constraints and Sharpe verified.")

# Verify original robust portfolios
print("\n=== ROBUST PORTFOLIO CONSTRAINTS ===")

for name, output in ablation_outputs.items():

    W = output["weights"]

    gross = W.abs().sum(axis=1)
    max_position = W.abs().max(axis=1)

    assert (gross <= MAX_GROSS + 1e-5).all()
    assert (max_position <= MAX_W + 1e-5).all()

    print(
        name,
        "| max gross:",
        round(gross.max(), 4),
        "| max position:",
        round(max_position.max(), 4)
    )

print("\nPORTFOLIO AUDIT COMPLETED")


=== PORTFOLIO AUDIT ===
              model  min_weight  max_budget_error  calculated_sharpe  reported_sharpe  sharpe_difference
        A: Standard    0.000000      1.800000e-07           0.341625         0.341625                0.0
B: Uncertainty only    0.000336      1.800000e-07           0.858297         0.858297                0.0
   C: Turnover only    0.000000      2.400000e-07           0.385506         0.385506                0.0
  D: Both penalties    0.002050      1.200000e-07           0.959333         0.959333                0.0

PASS: E2E portfolio constraints and Sharpe verified.

=== ROBUST PORTFOLIO CONSTRAINTS ===
Baseline | max gross: 1.5 | max position: 1.0
Robust only | max gross: 1.5 | max position: 0.7293
Turnover only | max gross: 1.5 | max position: 1.0
Both penalties | max gross: 1.5 | max position: 0.4512

PORTFOLIO AUDIT COMPLETED


# Check Model reproducibility

In [33]:

# EXPERIMENT 11C: REPRODUCIBILITY AUDIT

import numpy as np
import pandas as pd
import torch

print("=== REPRODUCIBILITY AUDIT ===")

repro_rows = []

for name, (rho, lam) in ABLATION10.items():

    # Refit using identical initialization and data
    model, history = fit_ablation10(rho, lam)

    model.eval()

    with torch.no_grad():
        new_pred = model(Xva)
        new_weights = portfolio_weights(new_pred).numpy()

    original_weights = outputs10[name]["weights"].to_numpy()

    max_difference = np.max(
        np.abs(new_weights - original_weights)
    )

    repro_rows.append({
        "model": name,
        "max_weight_difference": max_difference,
        "reproduced": max_difference < 1e-7
    })

repro = pd.DataFrame(repro_rows)

print(repro.to_string(index=False))

assert repro["reproduced"].all(), (
    "FAIL: Some models did not reproduce their weights"
)

print("\nPASS: All four end-to-end models reproduced.")

repro.to_csv(
    "experiment11_reproducibility.csv",
    index=False
)


=== REPRODUCIBILITY AUDIT ===
              model  max_weight_difference  reproduced
        A: Standard                    0.0        True
B: Uncertainty only                    0.0        True
   C: Turnover only                    0.0        True
  D: Both penalties                    0.0        True

PASS: All four end-to-end models reproduced.


In [35]:

# EXPERIMENT 11D: TIMESTAMP ALIGNMENT AUDIT

import pandas as pd
import numpy as np
import inspect

print("=== TIMESTAMP ALIGNMENT ===")

train_dates = pd.DatetimeIndex(SPLIT["train"])
val_dates = pd.DatetimeIndex(SPLIT["validation"])

print("\nTRAINING PERIOD")
print(train_dates.min(), "to", train_dates.max())

print("\nVALIDATION PERIOD")
print(val_dates.min(), "to", val_dates.max())

print("\nFIRST HISTORICAL RETURN DATES")
print(PAST.index[:5])

print("\nFIRST TARGET DATES")
print(TARGETS.index[:5])

print("\nFEATURES")
print(FEATURES)

date = val_dates[0]

print("\nEXAMPLE DECISION DATE:", date)

print("\nHISTORICAL RETURNS USED")
print(PAST.loc[:date].tail(3).iloc[:, :5])

print("\nTARGET RETURNS")
print(TARGETS.loc[date, TICKERS[:5]])

print("\nFEATURE VALUES")
print(
    panel.loc[date]
    .reindex(TICKERS)[FEATURES]
    .head()
)

print("\nCOVARIANCE FUNCTION SOURCE")
try:
    print(inspect.getsource(rolling_cov))
except (OSError, TypeError):
    print("Source unavailable; inspect the original notebook cell.")

print("\nAUDIT DIAGNOSTICS COMPLETED")


=== TIMESTAMP ALIGNMENT ===

TRAINING PERIOD
2009-12-25 00:00:00 to 2018-12-21 00:00:00

VALIDATION PERIOD
2018-12-28 00:00:00 to 2021-12-24 00:00:00

FIRST HISTORICAL RETURN DATES
DatetimeIndex(['2009-12-25', '2010-01-01', '2010-01-08', '2010-01-15', '2010-01-22'], dtype='datetime64[ns]', name='decision_date', freq='W-FRI')

FIRST TARGET DATES
DatetimeIndex(['2009-12-25', '2010-01-01', '2010-01-08', '2010-01-15', '2010-01-22'], dtype='datetime64[ns]', name='decision_date', freq='W-FRI')

FEATURES
['ret_1w', 'mom_4w', 'mom_12w', 'mom_26w', 'vol_12w', 'vol_26w', 'spy_mom_4w', 'spy_mom_12w', 'spy_vol_12w']

EXAMPLE DECISION DATE: 2018-12-28 00:00:00

HISTORICAL RETURNS USED
ticker              SPY       XLK       XLF       XLE       XLV
decision_date                                                  
2018-12-14    -0.011761 -0.000464 -0.034647 -0.030904 -0.017668
2018-12-21    -0.070505 -0.079186 -0.053956 -0.087859 -0.066375
2018-12-28     0.029290  0.037162  0.035103  0.016753  0.031084

In [36]:

# EXPERIMENT 11E: CHECK FOR TARGET LEAKAGE

common_dates = PAST.index.intersection(TARGETS.index)
common_tickers = [
    t for t in TICKERS
    if t in PAST.columns and t in TARGETS.columns
]

past = PAST.loc[common_dates, common_tickers]
target = TARGETS.loc[common_dates, common_tickers]

same_date_corr = past.stack().corr(target.stack())

same_date_equal = np.isclose(
    past.to_numpy(),
    target.to_numpy(),
    rtol=1e-8,
    atol=1e-10,
    equal_nan=False
).mean()

print("Same-date return correlation:",
      round(same_date_corr, 6))

print("Fraction of identical observations:",
      round(same_date_equal, 6))

print("\nExample:")
ticker = common_tickers[0]

print(pd.DataFrame({
    "PAST": past[ticker].head(5),
    "TARGET": target[ticker].head(5)
}))

if same_date_equal > 0.99:
    print("\nWARNING: PAST and TARGETS are almost identical.")
    print("Inspect timing before trusting the backtest.")
else:
    print("\nPAST and TARGETS are not identical.")
    print("Timing still requires source-code verification.")


Same-date return correlation: -0.082786
Fraction of identical observations: 0.0

Example:
                   PAST    TARGET
decision_date                    
2009-12-25     0.020596 -0.009246
2010-01-01    -0.009246  0.028087
2010-01-08     0.028087 -0.008117
2010-01-15    -0.008117 -0.038983
2010-01-22    -0.038983 -0.016665

PAST and TARGETS are not identical.
Timing still requires source-code verification.


In [37]:

# EXPERIMENT 11F: VERIFY ONE-WEEK TARGET SHIFT

import numpy as np
import pandas as pd

dates = PAST.index.intersection(TARGETS.index)
tickers = [t for t in TICKERS
           if t in PAST.columns and t in TARGETS.columns]

past = PAST.loc[dates, tickers]
target = TARGETS.loc[dates, tickers]

# Compare target at t with historical return at t+1
next_week = past.shift(-1)

mask = (
    target.notna() &
    next_week.notna()
)

difference = (
    target - next_week
).where(mask)

max_error = difference.abs().max().max()
mean_error = difference.abs().stack().mean()

print("=== TARGET ALIGNMENT AUDIT ===")
print("Maximum absolute error:", max_error)
print("Mean absolute error:", mean_error)

print("\nExample: SPY")
print(pd.DataFrame({
    "PAST_t": past["SPY"],
    "TARGET_t": target["SPY"],
    "PAST_t_plus_1": next_week["SPY"]
}).head(8))

if max_error < 1e-10:
    print("\nPASS: Targets match next-week historical returns.")
else:
    print("\nCHECK: Targets do not exactly match next-week returns.")
    print("Inspect target construction and date alignment.")


=== TARGET ALIGNMENT AUDIT ===
Maximum absolute error: 0.0
Mean absolute error: 0.0

Example: SPY
                 PAST_t  TARGET_t  PAST_t_plus_1
decision_date                                   
2009-12-25     0.020596 -0.009246      -0.009246
2010-01-01    -0.009246  0.028087       0.028087
2010-01-08     0.028087 -0.008117      -0.008117
2010-01-15    -0.008117 -0.038983      -0.038983
2010-01-22    -0.038983 -0.016665      -0.016665
2010-01-29    -0.016665 -0.006797      -0.006797
2010-02-05    -0.006797  0.012938       0.012938
2010-02-12     0.012938  0.028693       0.028693

PASS: Targets match next-week historical returns.


In [38]:

# EXPERIMENT 11G: FEATURE SCALING AUDIT

import inspect
import numpy as np
import pandas as pd

print("=== FEATURE STANDARDISATION AUDIT ===")

print("\n1. Feature ranges in training and validation")

for feature in FEATURES:
    train_values = panel.loc[
        panel.index.get_level_values(0).isin(SPLIT["train"]),
        feature
    ].to_numpy()

    val_values = panel.loc[
        panel.index.get_level_values(0).isin(SPLIT["validation"]),
        feature
    ].to_numpy()

    print(
        f"{feature:15s}",
        f"train_mean={train_values.mean(): .5f}",
        f"train_sd={train_values.std():.5f}",
        f"val_mean={val_values.mean(): .5f}"
    )

print("\n2. Source code for feature processing")

for function_name in [
    "make_data",
    "fit_e2e",
    "load_panels"
]:
    fn = globals().get(function_name)

    if fn is None:
        print(function_name, ": not found")
        continue

    print(f"\n--- {function_name} ---")

    try:
        print(inspect.getsource(fn))
    except (OSError, TypeError):
        print("Source unavailable; inspect notebook definition.")

print("\n3. Search notebook for preprocessing")

try:
    from IPython import get_ipython
    history = get_ipython().history_manager.input_hist_raw

    keywords = [
        "StandardScaler",
        "fit_transform",
        "scaler.fit",
        "mean(axis",
        "std(axis",
        "Xtr",
        "Xva"
    ]

    for i, cell in enumerate(history):
        matches = [
            k for k in keywords
            if k in cell
        ]

        if matches and i > 0:
            print(
                f"Cell {i}:",
                ", ".join(matches)
            )

except Exception as e:
    print("Could not inspect notebook history:", e)

print("\nAUDIT COMPLETE — SOURCE REVIEW REQUIRED")


=== FEATURE STANDARDISATION AUDIT ===

1. Feature ranges in training and validation
ret_1w          train_mean= 0.00175 train_sd=0.02151 val_mean= 0.00371
mom_4w          train_mean= 0.00731 train_sd=0.03986 val_mean= 0.01331
mom_12w         train_mean= 0.02269 train_sd=0.06368 val_mean= 0.03739
mom_26w         train_mean= 0.05353 train_sd=0.09212 val_mean= 0.07389
vol_12w         train_mean= 0.01919 train_sd=0.00983 val_mean= 0.02564
vol_26w         train_mean= 0.01967 train_sd=0.00883 val_mean= 0.02721
spy_mom_4w      train_mean= 0.00947 train_sd=0.03482 val_mean= 0.01784
spy_mom_12w     train_mean= 0.02969 train_sd=0.05311 val_mean= 0.04926
spy_vol_12w     train_mean= 0.01768 train_sd=0.00825 val_mean= 0.02438

2. Source code for feature processing

--- make_data ---
def make_data(dates):
    X, Y, C = [], [], []
    for date in dates:
        x = panel.loc[date].reindex(TICKERS)[FEATURES].to_numpy(float)
        X.append((x - mean) / std)
        Y.append(TARGETS.loc[date, TICKERS]

In [39]:

# EXPERIMENT 11H: VERIFY SCALER FITTING

import numpy as np

print("=== SCALER VERIFICATION ===")

print("Current mean:", mean)
print("Current std:", std)

# Full training sample
train_dates = SPLIT["train"]

train_features = (
    panel.loc[
        panel.index.get_level_values(0).isin(train_dates),
        FEATURES
    ]
    .to_numpy(float)
)

full_mean = train_features.mean(axis=0)
full_std = train_features.std(axis=0)

print("\nMatches full training mean:",
      np.allclose(mean, full_mean))

print("Matches full training std:",
      np.allclose(std, full_std))

# Internal training subset (first 393 usable weeks)
internal_dates = pd.DatetimeIndex(SPLIT["train"])[-len(Xtr):-26]

internal_features = (
    panel.loc[
        panel.index.get_level_values(0).isin(internal_dates),
        FEATURES
    ]
    .to_numpy(float)
)

internal_mean = internal_features.mean(axis=0)
internal_std = internal_features.std(axis=0)

print("\nMatches internal training mean:",
      np.allclose(mean, internal_mean))

print("Matches internal training std:",
      np.allclose(std, internal_std))


=== SCALER VERIFICATION ===
Current mean: [0.00174935 0.00730876 0.02269213 0.05353206 0.01918936 0.01966572
 0.0094673  0.02969263 0.01767771]
Current std: [0.02150821 0.03985828 0.06368191 0.09212334 0.00982936 0.00883219
 0.03481972 0.05310972 0.00824703]

Matches full training mean: True
Matches full training std: True

Matches internal training mean: False
Matches internal training std: False


In [40]:

# EXPERIMENT 12A: LEAKAGE-FREE FEATURE SCALING
# Reuses Experiment 9 definitions

import numpy as np
import torch
import pandas as pd

usable_dates = pd.DatetimeIndex(SPLIT["train"])[-len(Xtr):]
internal_train_dates = usable_dates[:-26]
internal_holdout_dates = usable_dates[-26:]

def feature_array(dates):
    return np.stack([
        panel.loc[date].reindex(TICKERS)[FEATURES].to_numpy(float)
        for date in dates
    ])

X_internal_raw = feature_array(internal_train_dates)
X_holdout_raw = feature_array(internal_holdout_dates)

# Fit ONLY on internal training observations
internal_mean = X_internal_raw.reshape(-1, len(FEATURES)).mean(axis=0)
internal_std = X_internal_raw.reshape(-1, len(FEATURES)).std(axis=0)
internal_std = np.maximum(internal_std, 1e-10)

X_internal = torch.tensor(
    (X_internal_raw - internal_mean) / internal_std,
    dtype=torch.float32
)

X_holdout = torch.tensor(
    (X_holdout_raw - internal_mean) / internal_std,
    dtype=torch.float32
)

assert len(X_internal) == 393
assert len(X_holdout) == 26

print("Internal training weeks:", len(X_internal))
print("Internal holdout weeks:", len(X_holdout))
print("Scaler fitted only on internal training: PASS")


Internal training weeks: 393
Internal holdout weeks: 26
Scaler fitted only on internal training: PASS


In [41]:

# EXPERIMENT 12B: CORRECTED EPOCH SELECTION

import torch.nn.functional as F

def fit_corrected(mode, epochs):
    torch.manual_seed(42)
    model = PortfolioModel()

    with torch.no_grad():
        model.linear.weight.zero_()
        model.linear.bias.zero_()

    opt = torch.optim.Adam(model.parameters(), lr=0.0001)

    for _ in range(epochs):
        opt.zero_grad()

        if mode == "MSE":
            loss = F.mse_loss(
                model(X_internal), Ytr[:-26]
            )
        else:
            rho = RHO_E2E if mode == "Robust decision-aware" else 0.0
            loss = improved_decision_loss(
                model,
                X_internal,
                Ytr[:-26],
                Ctr[:-26],
                rho=rho,
                lam=LAMBDA_E2E
            )

        loss.backward()
        opt.step()

    return model


rows = []

for mode in MODES:
    for epochs in [25, 50, 100, 200]:
        model = fit_corrected(mode, epochs)

        with torch.no_grad():
            score = -improved_decision_loss(
                model,
                X_holdout,
                Ytr[-26:],
                Ctr[-26:],
                rho=RHO_E2E,
                lam=LAMBDA_E2E
            ).item()

        rows.append({
            "model": mode,
            "epochs": epochs,
            "holdout_utility": score
        })

corrected_selection = pd.DataFrame(rows)

best_epochs_corrected = (
    corrected_selection
    .loc[
        corrected_selection.groupby("model")[
            "holdout_utility"
        ].idxmax()
    ]
    .set_index("model")["epochs"]
    .to_dict()
)

print("CORRECTED EPOCH SELECTION")
print(corrected_selection.round(6).to_string(index=False))

print("\nSELECTED EPOCHS")
print(best_epochs_corrected)

corrected_selection.to_csv(
    "experiment12_corrected_selection.csv",
    index=False
)


CORRECTED EPOCH SELECTION
                model  epochs  holdout_utility
                  MSE      25        -0.004495
                  MSE      50        -0.004444
                  MSE     100        -0.004418
                  MSE     200        -0.004418
       Decision-aware      25        -0.004683
       Decision-aware      50        -0.004999
       Decision-aware     100        -0.005170
       Decision-aware     200        -0.005178
Robust decision-aware      25        -0.004228
Robust decision-aware      50        -0.004499
Robust decision-aware     100        -0.004419
Robust decision-aware     200        -0.004424

SELECTED EPOCHS
{'Decision-aware': 25, 'MSE': 200, 'Robust decision-aware': 25}


In [42]:

# EXPERIMENT 12C: CORRECTED FINAL VALIDATION

results12 = []

for mode in MODES:
    epochs = int(best_epochs_corrected[mode])

    # Existing function uses full training and its fitted scaler
    model = fit_e2e(
        mode, epochs, seed=42, full=True
    )

    with torch.no_grad():
        pred = model(Xva)
        weights = portfolio_weights(pred)

    P = pd.DataFrame(
        pred.numpy(),
        index=va,
        columns=TICKERS
    )

    W = pd.DataFrame(
        weights.numpy(),
        index=va,
        columns=TICKERS
    )

    metrics, series, wealth = evaluate_portfolio(
        W,
        TARGETS.loc[va, TICKERS],
        predictions=P,
        cost_bps=COST_BPS,
        allow_short=False,
        max_gross=1.0
    )

    results12.append({
        "model": mode,
        "epochs": epochs,
        "sharpe": metrics["sharpe_net"],
        "turnover": metrics["mean_weekly_turnover"],
        "annual_return": metrics["ann_return"],
        "drawdown": metrics["max_drawdown"]
    })

results12 = pd.DataFrame(results12)

print("LEAKAGE-CORRECTED VALIDATION RESULTS")
print(results12.round(6).to_string(index=False))

results12.to_csv(
    "experiment12_corrected_results.csv",
    index=False
)


LEAKAGE-CORRECTED VALIDATION RESULTS
                model  epochs   sharpe  turnover  annual_return  drawdown
                  MSE     200 0.880619  0.454313       0.188888 -0.372380
       Decision-aware      25 0.385506  0.418784       0.060837 -0.419853
Robust decision-aware      25 0.959333  0.114748       0.152542 -0.285360


In [43]:

# EXPERIMENT 12D: CORRECTED HOLDOUT UTILITY
# Run after Experiments 12A-12C

import numpy as np
import pandas as pd
import torch

def corrected_holdout_utility(
    model,
    X_train,
    Y_train,
    X_holdout,
    Y_holdout,
    C_holdout,
    rho=RHO_E2E,
    lam=LAMBDA_E2E
):
    model.eval()

    with torch.no_grad():
        # Last training-period portfolio
        w_previous = portfolio_weights(
            model(X_train[-1:])
        )[0]

        # Return earned during last training period
        r_previous = Y_train[-1]

        # Holdings drift before first holdout rebalance
        growth = 1.0 + torch.dot(
            w_previous, r_previous
        )

        drifted_previous = (
            w_previous * (1.0 + r_previous)
            / growth.clamp(min=1e-8)
        )

        # Holdout portfolio allocations
        w = portfolio_weights(
            model(X_holdout)
        )

        gross_return = (
            w * Y_holdout
        ).sum(dim=1)

        risk = torch.einsum(
            "bi,bij,bj->b",
            w, C_holdout, w
        )

        uncertainty = torch.linalg.vector_norm(
            w, dim=1
        )

        # Previous holdings for each holdout week
        previous_w = w[:-1]
        previous_r = Y_holdout[:-1]

        growth_inside = 1.0 + (
            previous_w * previous_r
        ).sum(dim=1)

        drifted_inside = (
            previous_w * (1.0 + previous_r)
            / growth_inside.unsqueeze(1).clamp(min=1e-8)
        )

        pretrade = torch.cat(
            [
                drifted_previous.unsqueeze(0),
                drifted_inside
            ],
            dim=0
        )

        turnover = torch.sqrt(
            (w - pretrade)**2 + 1e-10
        ).sum(dim=1)

        utility = (
            gross_return
            - 0.5 * RISK_GAMMA * risk
            - rho * uncertainty
            - lam * turnover
            - COST_E2E * turnover
        )

        return float(utility.mean().item())


In [44]:

# EXPERIMENT 12E: FINAL TRAINING-ONLY EPOCH SELECTION

rows12d = []

for mode in MODES:
    for epochs in [25, 50, 100, 200]:

        model = fit_corrected(
            mode, epochs
        )

        score = corrected_holdout_utility(
            model=model,
            X_train=X_internal,
            Y_train=Ytr[:-26],
            X_holdout=X_holdout,
            Y_holdout=Ytr[-26:],
            C_holdout=Ctr[-26:],
            rho=RHO_E2E,
            lam=LAMBDA_E2E
        )

        rows12d.append({
            "model": mode,
            "epochs": epochs,
            "holdout_utility": score
        })

selection12d = pd.DataFrame(rows12d)

best_epochs12d = (
    selection12d.loc[
        selection12d.groupby("model")[
            "holdout_utility"
        ].idxmax()
    ]
    .set_index("model")["epochs"]
    .to_dict()
)

print("=== CORRECTED HOLDOUT RESULTS ===")
print(selection12d.round(7).to_string(index=False))

print("\n=== FINAL SELECTED EPOCHS ===")
print(best_epochs12d)

selection12d.to_csv(
    "experiment12d_holdout_selection.csv",
    index=False
)


=== CORRECTED HOLDOUT RESULTS ===
                model  epochs  holdout_utility
                  MSE      25        -0.004462
                  MSE      50        -0.004411
                  MSE     100        -0.004386
                  MSE     200        -0.004386
       Decision-aware      25        -0.004658
       Decision-aware      50        -0.004974
       Decision-aware     100        -0.005148
       Decision-aware     200        -0.005155
Robust decision-aware      25        -0.004193
Robust decision-aware      50        -0.004464
Robust decision-aware     100        -0.004384
Robust decision-aware     200        -0.004389

=== FINAL SELECTED EPOCHS ===
{'Decision-aware': 25, 'MSE': 200, 'Robust decision-aware': 25}


In [45]:

# EXPERIMENT 12F: FINAL VALIDATION RESULTS

results12f = []

for mode in MODES:

    epochs = int(best_epochs12d[mode])

    model = fit_e2e(
        mode,
        epochs,
        seed=42,
        full=True
    )

    model.eval()

    with torch.no_grad():
        predictions = model(Xva)
        weights = portfolio_weights(predictions)

    P = pd.DataFrame(
        predictions.numpy(),
        index=va,
        columns=TICKERS
    )

    W = pd.DataFrame(
        weights.numpy(),
        index=va,
        columns=TICKERS
    )

    metrics, series, wealth = evaluate_portfolio(
        W,
        TARGETS.loc[va, TICKERS],
        predictions=P,
        cost_bps=COST_BPS,
        allow_short=False,
        max_gross=1.0
    )

    results12f.append({
        "model": mode,
        "epochs": epochs,
        "sharpe": metrics["sharpe_net"],
        "annual_return": metrics["ann_return"],
        "turnover": metrics["mean_weekly_turnover"],
        "drawdown": metrics["max_drawdown"]
    })

results12f = pd.DataFrame(results12f)

print("=== FINAL VALIDATION COMPARISON ===")
print(results12f.round(6).to_string(index=False))

results12f.to_csv(
    "experiment12f_final_validation.csv",
    index=False
)


=== FINAL VALIDATION COMPARISON ===
                model  epochs   sharpe  annual_return  turnover  drawdown
                  MSE     200 0.880619       0.188888  0.454313 -0.372380
       Decision-aware      25 0.385506       0.060837  0.418784 -0.419853
Robust decision-aware      25 0.959333       0.152542  0.114748 -0.285360


# Reprocessing Audit

In [46]:

# EXPERIMENT 13A: ROLLING-FOLD PREPROCESSING AUDIT

import numpy as np
import pandas as pd

print("=== ROLLING-FOLD PREPROCESSING AUDIT ===")

# Load saved fold schedule
fold_schedule = pd.read_csv(
    "walk_forward_fold_schedule.csv"
)

print("\nFOLD SCHEDULE COLUMNS:")
print(fold_schedule.columns.tolist())
print("\nFIRST ROWS:")
print(fold_schedule.head().to_string(index=False))

# Raw feature matrix for all training dates
train_dates = pd.DatetimeIndex(SPLIT["train"])

raw_features = (
    panel.loc[
        panel.index.get_level_values(0).isin(train_dates),
        FEATURES
    ]
    .to_numpy(dtype=float)
)

full_mean = raw_features.mean(axis=0)
full_std = raw_features.std(axis=0)

print("\nFULL-TRAINING FEATURE STATISTICS")
print("Mean:", np.round(full_mean, 6))
print("Std:", np.round(full_std, 6))

# Find candidate fold boundary columns
print("\nPOSSIBLE DATE COLUMNS:")
print([
    c for c in fold_schedule.columns
    if "date" in c.lower()
    or "start" in c.lower()
    or "end" in c.lower()
])

print("\nAUDIT SETUP COMPLETE")


=== ROLLING-FOLD PREPROCESSING AUDIT ===

FOLD SCHEDULE COLUMNS:
['fold', 'train_start_outcome', 'train_end_outcome', 'validation_start_outcome', 'validation_end_outcome']

FIRST ROWS:
 fold train_start_outcome train_end_outcome validation_start_outcome validation_end_outcome
    1          2013-12-27        2016-12-16               2016-12-23             2017-06-16
    2          2014-06-27        2017-06-16               2017-06-23             2017-12-15
    3          2014-12-26        2017-12-15               2017-12-22             2018-06-15
    4          2015-06-26        2018-06-15               2018-06-22             2018-12-14

FULL-TRAINING FEATURE STATISTICS
Mean: [0.001749 0.007309 0.022692 0.053532 0.019189 0.019666 0.009467 0.029693
 0.017678]
Std: [0.021508 0.039858 0.063682 0.092123 0.009829 0.008832 0.03482  0.05311
 0.008247]

POSSIBLE DATE COLUMNS:
['train_start_outcome', 'train_end_outcome', 'validation_start_outcome', 'validation_end_outcome']

AUDIT SETUP COMPLET

In [47]:

# EXPERIMENT 13B: INSPECT ORIGINAL TUNING CODE

import inspect

print("=== ORIGINAL MODEL SELECTION AUDIT ===")

candidate_functions = [
    "make_data",
    "fit_corrected",
    "fit_e2e",
    "rolling_cov",
    "evaluate_portfolio"
]

for name in candidate_functions:
    fn = globals().get(name)

    if fn is None:
        continue

    print(f"\n{'=' * 15} {name} {'=' * 15}")

    try:
        source = inspect.getsource(fn)
        print(source)

    except (OSError, TypeError):
        print("Source not available.")

print("\n=== SEARCH FOR SCALER OPERATIONS ===")

from IPython import get_ipython

history = get_ipython().history_manager.input_hist_raw

keywords = [
    "StandardScaler",
    "fit_transform",
    "scaler.fit",
    "mean =",
    "std =",
    "fold",
    "walk_forward"
]

for i, cell in enumerate(history):
    if i == 0:
        continue

    matches = [
        word for word in keywords
        if word in cell
    ]

    if matches:
        print(
            f"Cell {i}:",
            ", ".join(matches)
        )


=== ORIGINAL MODEL SELECTION AUDIT ===

=============== make_data ===============
def make_data(dates):
    X, Y, C = [], [], []
    for date in dates:
        x = panel.loc[date].reindex(TICKERS)[FEATURES].to_numpy(float)
        X.append((x - mean) / std)
        Y.append(TARGETS.loc[date, TICKERS].to_numpy(float))
        C.append(rolling_cov(PAST, date))
    return (
        torch.tensor(np.array(X), dtype=torch.float32),
        torch.tensor(np.array(Y), dtype=torch.float32),
        torch.tensor(np.array(C), dtype=torch.float32)
    )


=============== fit_corrected ===============
def fit_corrected(mode, epochs):
    torch.manual_seed(42)
    model = PortfolioModel()

    with torch.no_grad():
        model.linear.weight.zero_()
        model.linear.bias.zero_()

    opt = torch.optim.Adam(model.parameters(), lr=0.0001)

    for _ in range(epochs):
        opt.zero_grad()

        if mode == "MSE":
            loss = F.mse_loss(
                model(X_internal), Ytr[:-26]
     

In [48]:

# EXPERIMENT 13C: FINAL MODEL REPRODUCIBILITY

import torch
import numpy as np
import pandas as pd

print("=== FINAL MODEL REPRODUCIBILITY ===")

rows13 = []

for mode in MODES:

    epochs = int(best_epochs12d[mode])

    model1 = fit_e2e(
        mode, epochs, seed=42, full=True
    )

    model2 = fit_e2e(
        mode, epochs, seed=42, full=True
    )

    model1.eval()
    model2.eval()

    with torch.no_grad():
        w1 = portfolio_weights(
            model1(Xva)
        ).numpy()

        w2 = portfolio_weights(
            model2(Xva)
        ).numpy()

    max_difference = np.max(
        np.abs(w1 - w2)
    )

    rows13.append({
        "model": mode,
        "epochs": epochs,
        "max_weight_difference": max_difference,
        "reproducible": max_difference < 1e-7
    })

repro13 = pd.DataFrame(rows13)

print(repro13.to_string(index=False))

assert repro13["reproducible"].all()

repro13.to_csv(
    "experiment13_reproducibility.csv",
    index=False
)

print("\nPASS: Final models reproduce.")


=== FINAL MODEL REPRODUCIBILITY ===
                model  epochs  max_weight_difference  reproducible
                  MSE     200                    0.0          True
       Decision-aware      25                    0.0          True
Robust decision-aware      25                    0.0          True

PASS: Final models reproduce.


In [49]:

# EXPERIMENT 13E: FOLD-SPECIFIC SCALING AUDIT

import numpy as np
import pandas as pd

schedule = pd.read_csv(
    "walk_forward_fold_schedule.csv",
    parse_dates=[
        "train_start_outcome",
        "train_end_outcome",
        "validation_start_outcome",
        "validation_end_outcome"
    ]
)

dates = pd.DatetimeIndex(SPLIT["train"])
raw = np.stack([
    panel.loc[d].reindex(TICKERS)[FEATURES].to_numpy(float)
    for d in dates
])

records = []
fold_scalers = {}

for _, row in schedule.iterrows():
    fold = int(row["fold"])

    # A decision at t has an outcome at t+7 days
    outcome_dates = dates + pd.Timedelta(days=7)

    train_mask = (
        (outcome_dates >= row["train_start_outcome"]) &
        (outcome_dates <= row["train_end_outcome"])
    )

    val_mask = (
        (outcome_dates >= row["validation_start_outcome"]) &
        (outcome_dates <= row["validation_end_outcome"])
    )

    train_raw = raw[train_mask]
    val_raw = raw[val_mask]

    assert len(train_raw) > 0
    assert len(val_raw) > 0

    assert (
        outcome_dates[train_mask].max()
        < outcome_dates[val_mask].min()
    )

    # Fit only on this fold's training observations
    fold_mean = train_raw.mean(axis=(0, 1))
    fold_std = train_raw.std(axis=(0, 1))
    fold_std = np.maximum(fold_std, 1e-10)

    X_fold_train = (train_raw - fold_mean) / fold_std
    X_fold_val = (val_raw - fold_mean) / fold_std

    fold_scalers[fold] = {
        "mean": fold_mean,
        "std": fold_std,
        "train_dates": dates[train_mask],
        "validation_dates": dates[val_mask]
    }

    records.append({
        "fold": fold,
        "train_weeks": len(train_raw),
        "validation_weeks": len(val_raw),
        "train_scaled_mean":
            float(np.abs(X_fold_train.mean(axis=(0,1))).max()),
        "train_scaled_std_error":
            float(np.abs(X_fold_train.std(axis=(0,1))-1).max()),
        "different_from_full_scaler":
            not np.allclose(fold_mean, mean)
    })

audit13e = pd.DataFrame(records)

print("=== FOLD-SPECIFIC SCALING AUDIT ===")
print(audit13e.to_string(index=False))

audit13e.to_csv(
    "experiment13e_fold_scaling_audit.csv",
    index=False
)


=== FOLD-SPECIFIC SCALING AUDIT ===
 fold  train_weeks  validation_weeks  train_scaled_mean  train_scaled_std_error  different_from_full_scaler
    1          156                26       1.553743e-15            6.661338e-16                        True
    2          156                26       1.892503e-15            4.440892e-16                        True
    3          156                26       1.189931e-15            4.440892e-16                        True
    4          156                26       1.093143e-15            8.881784e-16                        True


In [50]:

# EXPERIMENT 13F: INSPECT ORIGINAL FOLD IMPLEMENTATION

from IPython import get_ipython

history = get_ipython().history_manager.input_hist_raw

keywords = [
    "walk_forward_fold_schedule.csv",
    "experiment4_fold_results.csv",
    "tuning_folds.csv",
    "training_arch_lm_tests.csv",
    "StandardScaler",
    "fit_transform"
]

matches = []

for i, cell in enumerate(history):
    if i == 0:
        continue

    found = [
        word for word in keywords
        if word in cell
    ]

    if found:
        matches.append((i, found, cell))

print("=== ORIGINAL FOLD CODE LOCATIONS ===")

for i, found, cell in matches:
    print(f"\nCELL {i}")
    print("Keywords:", found)

    # Show the relevant source, excluding this audit cell
    if "EXPERIMENT 13F" not in cell:
        print(cell[:6000])
        print("\n" + "-" * 60)


=== ORIGINAL FOLD CODE LOCATIONS ===

CELL 9
Keywords: ['tuning_folds.csv']
import time, numpy as np, pandas as pd, itertools
tr = SPLIT['train']
# fold schedule: replicate Person 1 (156w train, 26w val, step 26, last 4) on decision dates
n=len(tr); folds=[]
for e in range(156, n-26+1, 26): folds.append((tr[e-156:e], tr[e:e+26]))
folds=folds[-4:]
for i,(a,b) in enumerate(folds,1):
    print(i, (a[0]+pd.Timedelta(days=7)).date(), (a[-1]+pd.Timedelta(days=7)).date(), (b[0]+pd.Timedelta(days=7)).date(), (b[-1]+pd.Timedelta(days=7)).date())
alphas=[0,10,100,1000,3000,10000,30000,100000]
gammas=[1,2.5,5,10,25,50,100]
rows=[]; t0=time.time()
for a in alphas:
    for k,(ftr,fva) in enumerate(folds,1):
        m=PooledLinearForecaster(a).fit(panel, ftr)
        pred=m.predict(panel, fva)
        act=TARGETS.loc[fva]
        mse=float(((pred-act)**2).to_numpy().mean())
        for g in gammas:
            s,_,_,W=run_strategy(pred, TARGETS, PAST, fva, g)
            rows.append(dict(alpha=a,gam

In [52]:

# EXPERIMENT 13G: RECOVER ORIGINAL FORECASTER CODE

from IPython import get_ipython

history = get_ipython().history_manager.input_hist_raw

print("=== POOLED LINEAR FORECASTER SOURCE ===")

found = False

for i, cell in enumerate(history):
    if "class PooledLinearForecaster" in cell:
        print(f"\nFOUND IN CELL {i}")
        print(cell)
        found = True

if not found:
    print("Class definition not found in execution history.")
    print("Search the notebook for: class PooledLinearForecaster")

print("\n=== FOLD SCALER COMPARISON ===")

print(
    audit13e[
        [
            "fold",
            "train_weeks",
            "validation_weeks",
            "different_from_full_scaler"
        ]
    ].to_string(index=False)
)


=== POOLED LINEAR FORECASTER SOURCE ===

FOUND IN CELL 3
"""QF209 A4 - Person 2: OLS/ridge predict-then-optimise baseline.
Uses only train_panel.csv and validation_panel.csv (no test labels).
"""
import numpy as np, pandas as pd
from scipy.optimize import minimize

TICKERS = ['SPY','XLK','XLF','XLE','XLV','XLI','XLY','XLP','XLU','XLB',
           'QQQ','IWM','EFA','EEM','TLT','IEF','LQD','HYG','GLD','VNQ']
FEATURES = ['ret_1w','mom_4w','mom_12w','mom_26w','vol_12w','vol_26w',
            'spy_mom_4w','spy_mom_12w','spy_vol_12w']
MAX_GROSS, MAX_W, COST_BPS = 1.5, 1.0, 5.0
COV_WINDOW = 156  # weeks of past realised returns for the covariance estimate


def load_panels(train_path='train_panel.csv', val_path='validation_panel.csv'):
    out = {}
    for name, p in [('train', train_path), ('validation', val_path)]:
        df = pd.read_csv(p, parse_dates=['decision_date', 'target_end_date'])
        out[name] = df.set_index(['decision_date', 'ticker']).sort_index()
    panel = pd.concat([ou

In [53]:

import pandas as pd

panel, SPLIT, TARGETS, PAST = load_panels()

schedule = pd.read_csv("walk_forward_fold_schedule.csv")

print("=== WALK-FORWARD LABEL LEAKAGE AUDIT ===")

results = []

for _, row in schedule.iterrows():
    train_end = pd.Timestamp(row["train_end_outcome"])
    validation_start = pd.Timestamp(row["validation_start_outcome"])

    train_rows = panel.loc[
        panel.index.get_level_values("decision_date") <= train_end
    ]

    latest_target_end = train_rows["target_end_date"].max()

    safe = latest_target_end <= validation_start

    results.append({
        "fold": row["fold"],
        "train_end_decision": train_end,
        "latest_training_target_end": latest_target_end,
        "validation_start": validation_start,
        "no_label_leakage": safe
    })

audit_labels = pd.DataFrame(results)

print(audit_labels.to_string(index=False))

if audit_labels["no_label_leakage"].all():
    print("\nPASS: All folds satisfy target-date separation.")
else:
    print("\nWARNING: At least one fold requires closer inspection.")


=== WALK-FORWARD LABEL LEAKAGE AUDIT ===
 fold train_end_decision latest_training_target_end validation_start  no_label_leakage
    1         2016-12-16                 2016-12-23       2016-12-23              True
    2         2017-06-16                 2017-06-23       2017-06-23              True
    3         2017-12-15                 2017-12-22       2017-12-22              True
    4         2018-06-15                 2018-06-22       2018-06-22              True

PASS: All folds satisfy target-date separation.


In [54]:

# EXPERIMENT 13H: CHECK ACTUAL WALK-FORWARD IMPLEMENTATION

from IPython import get_ipython

history = get_ipython().history_manager.input_hist_raw

print("=== WALK-FORWARD TRAINING CODE ===")

for i, cell in enumerate(history):
    if (
        "walk_forward_fold_schedule" in cell
        or "tuning_folds.csv" in cell
    ):
        if "def " in cell or ".fit(" in cell:
            print(f"\nCELL {i}")
            print(cell[:7000])


=== WALK-FORWARD TRAINING CODE ===

CELL 9
import time, numpy as np, pandas as pd, itertools
tr = SPLIT['train']
# fold schedule: replicate Person 1 (156w train, 26w val, step 26, last 4) on decision dates
n=len(tr); folds=[]
for e in range(156, n-26+1, 26): folds.append((tr[e-156:e], tr[e:e+26]))
folds=folds[-4:]
for i,(a,b) in enumerate(folds,1):
    print(i, (a[0]+pd.Timedelta(days=7)).date(), (a[-1]+pd.Timedelta(days=7)).date(), (b[0]+pd.Timedelta(days=7)).date(), (b[-1]+pd.Timedelta(days=7)).date())
alphas=[0,10,100,1000,3000,10000,30000,100000]
gammas=[1,2.5,5,10,25,50,100]
rows=[]; t0=time.time()
for a in alphas:
    for k,(ftr,fva) in enumerate(folds,1):
        m=PooledLinearForecaster(a).fit(panel, ftr)
        pred=m.predict(panel, fva)
        act=TARGETS.loc[fva]
        mse=float(((pred-act)**2).to_numpy().mean())
        for g in gammas:
            s,_,_,W=run_strategy(pred, TARGETS, PAST, fva, g)
            rows.append(dict(alpha=a,gamma=g,fold=k,mse=mse,r2_zero=r2_oo

## Final Models

In [55]:

# EXPERIMENT 14A — FREEZE FINAL MODEL SELECTION
# Run BEFORE opening or evaluating test data.

import pandas as pd
import json
from datetime import datetime, timezone

FINAL_CONFIG = {
    "primary_model": "Robust decision-aware",
    "primary_epochs": 25,
    "primary_rho": float(RHO_E2E),
    "primary_lambda": float(LAMBDA_E2E),

    "secondary_model": "Ridge + robust mean-variance",
    "secondary_rho": 0.005,
    "secondary_lambda": 0.0,

    "seed": 42,
    "selection_basis": "Training folds and 2018-2021 validation",
    "test_evaluated": False
}

# Protect against accidentally overwriting the freeze
from pathlib import Path
freeze_path = Path("final_model_freeze.json")

if freeze_path.exists():
    raise FileExistsError(
        "Model already frozen. Do not overwrite it."
    )

with open(freeze_path, "w") as f:
    json.dump(FINAL_CONFIG, f, indent=2)

print("=== FINAL MODELS FROZEN ===")
print(json.dumps(FINAL_CONFIG, indent=2))


=== FINAL MODELS FROZEN ===
{
  "primary_model": "Robust decision-aware",
  "primary_epochs": 25,
  "primary_rho": 0.005,
  "primary_lambda": 0.0005,
  "secondary_model": "Ridge + robust mean-variance",
  "secondary_rho": 0.005,
  "secondary_lambda": 0.0,
  "seed": 42,
  "selection_basis": "Training folds and 2018-2021 validation",
  "test_evaluated": false
}


In [56]:
# EXPERIMENT 14B — DOCUMENT FINAL SELECTION

import pandas as pd
from datetime import datetime, timezone

entry = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "Final model selection",
    "primary": "Robust decision-aware",
    "primary_validation_sharpe": 0.959333,
    "secondary": "Robust mean-variance",
    "secondary_validation_sharpe": 0.6771,
    "decision": "Freeze models before untouched test",
    "reason": (
        "Primary selected for risk-adjusted performance, "
        "turnover reduction and drawdown control. "
        "Secondary retained to assess optimiser robustness."
    )
}

pd.DataFrame([entry]).to_csv(
    "final_selection_log.csv",
    index=False
)

print("Final selection documented.")


Final selection documented.


In [57]:

# EXPERIMENT 14C — CHECK TEST DATA AVAILABILITY

import os
import pandas as pd

print("=== AVAILABLE DATA FILES ===")

for filename in sorted(os.listdir(".")):
    if filename.endswith((".csv", ".parquet", ".pkl")):
        print(filename)

print("\n=== TRAINING AND VALIDATION DATES ===")

panel, SPLIT, TARGETS, PAST = load_panels()

for name, dates in SPLIT.items():
    print(
        name,
        "from", dates.min().date(),
        "to", dates.max().date(),
        "weeks:", len(dates)
    )

print("\n=== TEST FILE CANDIDATES ===")

test_files = [
    f for f in os.listdir(".")
    if "test" in f.lower()
    and f.endswith((".csv", ".parquet"))
]

print(test_files if test_files else "No test file found")


=== AVAILABLE DATA FILES ===
cost_aware_results.csv
experiment10_ablation.csv
experiment10_effects.csv
experiment11_reproducibility.csv
experiment12_corrected_results.csv
experiment12_corrected_selection.csv
experiment12d_holdout_selection.csv
experiment12f_final_validation.csv
experiment13_reproducibility.csv
experiment13e_fold_scaling_audit.csv
experiment3_internal_selection.csv
experiment3_results.csv
experiment4_fold_results.csv
experiment4_tuning_summary.csv
experiment4_validation_comparison.csv
experiment5_ablation.csv
experiment6_cost_stress.csv
experiment7_bootstrap.csv
experiment8_portfolio_stability.csv
experiment9_results.csv
experiment9_training_selection.csv
final_selection_log.csv
iteration_log.csv
robust_experiment_results.csv
store.pkl
train_panel.csv
training_arch_lm_tests.csv
training_correlations.csv
tuning_folds.csv
validation_benchmark_metrics.csv
validation_mse_vs_sharpe_grid.csv
validation_panel.csv
validation_results_person2.csv
walk_forward_fold_schedule.csv

=

In [58]:

import json
from pathlib import Path

for path in Path(".").glob("*.ipynb"):
    notebook = json.loads(path.read_text())
    print(f"\nNOTEBOOK: {path.name}")

    for i, cell in enumerate(notebook.get("cells", [])):
        source = "".join(cell.get("source", []))
        if any(term in source.lower() for term in
               ["test_panel", "test dataset", "2022-",
                "download", "yfinance"]):
            print(f"\nCell {i}:")
            print(source[:1200])
